# ECG GNN Pipeline — Joint ANN + PDC Project

**Authors**: Uroosh Kamran (23i-0035), Afroz Talha (23i-2539)  
**Subject**: Artificial Neural Networks + Parallel & Distributed Computing

**Datasets**: MIT-BIH Arrhythmia Database · PTB-XL ECG Database

---
### How to run
1. Fill in your paths in **Cell 1** (replace `<<<REPLACE_ME>>>`)
2. Run all cells top to bottom using **Run All**
3. Or run cells individually — each section is self-contained

### Install dependencies first
```bash
pip install torch torchvision torch_geometric wfdb
pip install opencv-python-headless scikit-learn joblib tqdm pandas numpy matplotlib
```


---
## Cell — Module docstring & how-to-run guide


In [2]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [1]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
print(f"RAM freed. GPU memory: {torch.cuda.memory_allocated()/1e6:.1f} MB used")

RAM freed. GPU memory: 0.0 MB used


In [2]:
import shutil, os

dirs_to_clear = [
    "E:/ecg_projectt/ptbxl/images",
    "E:/ecg_projectt/ptbxl/edge_filtered", 
    "E:/ecg_projectt/ptbxl/graphs",
    
]

for d in dirs_to_clear:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Cleared: {d}")
    os.makedirs(d, exist_ok=True)
    print(f"Recreated: {d}")

print("Done!")

Recreated: E:/ecg_projectt/ptbxl/images
Recreated: E:/ecg_projectt/ptbxl/edge_filtered
Recreated: E:/ecg_projectt/ptbxl/graphs
Done!


In [3]:
# ==========================================================================
# Cell — Module docstring & how-to-run guide
# ==========================================================================

"""
=============================================================================
ECG GNN Pipeline — Unified Script
Joint ANN + PDC Project

ANN angle  : CNN node-feature extraction → GNN graph classification
             (GraphConv / GCN / GAT / GATv2) on MIT-BIH and PTB-XL datasets
PDC angle  : GPU acceleration, parallel preprocessing (joblib / multiprocessing),
             parallel dataset pipelines, checkpointing, Amdahl/Gustafson analysis

Datasets
  MIT-BIH Arrhythmia Database  https://physionet.org/content/mitdb/1.0.0/
  PTB-XL ECG Database          https://physionet.org/content/ptb-xl/1.0.3/

Authors : Uroosh Kamran (23i-0035), Afroz Talha (23i-2539)
Subject : Artificial Neural Networks + Parallel & Distributed Computing
=============================================================================

HOW TO RUN
----------
  python ecg_gnn_pipeline.py                          # full run, both datasets
  python ecg_gnn_pipeline.py --dataset mitbih         # MIT-BIH only
  python ecg_gnn_pipeline.py --dataset ptbxl          # PTB-XL only
  python ecg_gnn_pipeline.py --skip-download          # skip download if already on disk
  python ecg_gnn_pipeline.py --layer GAT              # choose GNN layer
  python ecg_gnn_pipeline.py --epochs 50              # change epoch count

REQUIRED PACKAGES
-----------------
  pip install torch torchvision torch_geometric wfdb
  pip install opencv-python-headless scikit-learn joblib tqdm pandas numpy matplotlib
"""

'\n=============================================================================\nECG GNN Pipeline — Unified Script\nJoint ANN + PDC Project\n\nANN angle  : CNN node-feature extraction → GNN graph classification\n             (GraphConv / GCN / GAT / GATv2) on MIT-BIH and PTB-XL datasets\nPDC angle  : GPU acceleration, parallel preprocessing (joblib / multiprocessing),\n             parallel dataset pipelines, checkpointing, Amdahl/Gustafson analysis\n\nDatasets\n  MIT-BIH Arrhythmia Database  https://physionet.org/content/mitdb/1.0.0/\n  PTB-XL ECG Database          https://physionet.org/content/ptb-xl/1.0.3/\n\nAuthors : Uroosh Kamran (23i-0035), Afroz Talha (23i-2539)\nSubject : Artificial Neural Networks + Parallel & Distributed Computing\n=============================================================================\n\nHOW TO RUN\n----------\n  python ecg_gnn_pipeline.py                          # full run, both datasets\n  python ecg_gnn_pipeline.py --dataset mitbih         # MIT-

---
## Cell 0 — Imports & environment check


In [4]:
# ==========================================================================
# Cell 0 — Imports & environment check
# ==========================================================================

# =============================================================================
# SECTION 0 — IMPORTS
# =============================================================================
import os
import sys
import gc
import ast
import glob
import json
import time
import shutil
import random
import argparse
import traceback
import warnings
import multiprocessing
from collections import Counter
from datetime import datetime
from pathlib import Path

from scipy import signal as scipy_signal

import numpy as np
import pandas as pd
import cv2
import wfdb
import matplotlib
matplotlib.use('Agg')   # headless — no display required on GPU server
import matplotlib.pyplot as plt

from tqdm import tqdm
from joblib import Parallel, delayed       # PDC: shared-memory data parallelism
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import NearestNeighbors

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as pyg_nn
from torch_geometric.nn import GraphConv, GCNConv, GATConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader, ImbalancedSampler
from torch_geometric.data import InMemoryDataset, Data as PyGData
from torch_geometric.io import read_tu_data

import os.path as osp

warnings.filterwarnings('ignore')

print("[DEBUG] All imports successful.")
print(f"[DEBUG] Python      : {sys.version}")
print(f"[DEBUG] PyTorch     : {torch.__version__}")
print(f"[DEBUG] CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[DEBUG] GPU name    : {torch.cuda.get_device_name(0)}")
    print(f"[DEBUG] GPU memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"[DEBUG] CPU cores   : {multiprocessing.cpu_count()}")

c:\Users\Afroz\Downloads\anaconda\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[DEBUG] All imports successful.
[DEBUG] Python      : 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]
[DEBUG] PyTorch     : 2.11.0+cu126
[DEBUG] CUDA avail  : True
[DEBUG] GPU name    : NVIDIA GeForce RTX 3050 Laptop GPU
[DEBUG] GPU memory  : 4.3 GB
[DEBUG] CPU cores   : 16


---
## Cell 1 — Path configuration (FILL IN YOUR PATHS HERE)


In [5]:
# ==========================================================================
# Cell 1 — Path configuration (FILL IN YOUR PATHS HERE)
# ==========================================================================

# =============================================================================
# SECTION 1 — PATH CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# INSTRUCTIONS: Replace every  <<<REPLACE_ME>>>  with the actual path on your
#               university GPU system.  Example values are shown in comments.
# =============================================================================

# ── Root directory for ALL pipeline outputs (graphs, checkpoints, results) ──
# Example: "/home/s23i0035/ecg_research"  or  "/scratch/ecg_research"

BASE_DIR = "E:/ecg_projectt"

# ── Where raw dataset files will be downloaded / already exist ──────────────
# Example: "/data/datasets"  or  "/home/s23i0035/datasets"
DATASETS_ROOT = "E:/ecg_projectt/data"

# Derived paths — do NOT change these, they are computed from the two above
MITBIH_RAW_DIR   = osp.join(DATASETS_ROOT, "mitbih")
PTBXL_RAW_DIR    = osp.join(DATASETS_ROOT, "ptb-xl")

MITBIH_IMG_DIR   = osp.join(BASE_DIR, "mitbih", "images")
MITBIH_EDGE_DIR  = osp.join(BASE_DIR, "mitbih", "edge_filtered")
MITBIH_GRAPH_DIR = osp.join(BASE_DIR, "mitbih", "graphs")

PTBXL_IMG_DIR    = osp.join(BASE_DIR, "ptbxl", "images")
PTBXL_EDGE_DIR   = osp.join(BASE_DIR, "ptbxl", "edge_filtered")
PTBXL_GRAPH_DIR  = osp.join(BASE_DIR, "ptbxl", "graphs")

CHECKPOINT_DIR   = osp.join(BASE_DIR, "checkpoints")
RESULTS_DIR      = osp.join(BASE_DIR, "results")

def _validate_paths():
    """Called at startup. Fails early and clearly if placeholders were not replaced."""
    print("\n[DEBUG] ── Path validation ──────────────────────────────────")
    errors = []
    if "<<<REPLACE_ME>>>" in BASE_DIR:
        errors.append("  BASE_DIR is still a placeholder — set it to your output folder.")
    if "<<<REPLACE_ME>>>" in DATASETS_ROOT:
        errors.append("  DATASETS_ROOT is still a placeholder — set it to your datasets folder.")
    if errors:
        print("[ERROR] You have un-filled path placeholders:")
        for e in errors:
            print(e)
        print("[ERROR] Open this script and replace every <<<REPLACE_ME>>> before running.")
        sys.exit(1)
    print(f"[DEBUG] BASE_DIR        : {BASE_DIR}")
    print(f"[DEBUG] DATASETS_ROOT   : {DATASETS_ROOT}")
    print(f"[DEBUG] CHECKPOINT_DIR  : {CHECKPOINT_DIR}")
    print(f"[DEBUG] RESULTS_DIR     : {RESULTS_DIR}")
    print("[DEBUG] Path validation passed.\n")

---
## Cell 2 — Global config (hyperparameters, dataset settings)


In [6]:
# ==========================================================================
# Cell 2 — Global config (hyperparameters, dataset settings)
# ==========================================================================

# =============================================================================
# SECTION 2 — GLOBAL CONFIG
# =============================================================================

# ── GPU / device ──────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[DEBUG] Training device : {DEVICE}")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Parallelism (PDC) ─────────────────────────────────────────────────────────
# Number of parallel CPU workers for preprocessing.
# -1  = use ALL available CPU cores (recommended on GPU server)
#  4  = safe default if the server has shared users
N_JOBS = 2
# DataLoader workers — 0 is safest if you hit shared-memory errors on the server
NUM_WORKERS = 0

# ── MIT-BIH config ────────────────────────────────────────────────────────────
MITBIH_IMG_SIZE    = (64, 64)
MITBIH_BRIGHTNESS  = 128
MITBIH_PATCH_SIZE  = 7
MITBIH_CNN_DIM     = 32
MITBIH_LABELS = {
    'N': '0', 'L': '1', 'R': '2', 'A': '3', 'V': '4'
}
MITBIH_REVERT = {v: k for k, v in MITBIH_LABELS.items()}
# How many MIT-BIH records to process. Set to None to process ALL 48.
MITBIH_MAX_RECORDS = 20

# Maximum beats to keep PER CLASS for MIT-BIH (stratified)
MITBIH_MAX_BEATS_PER_CLASS = 500   # 5 classes x 500 = 2,500 images total

# ── PTB-XL config ─────────────────────────────────────────────────────────────
PTBXL_IMG_SIZE     = (224, 224)
PTBXL_EDGE_SIZE    = 112
PTBXL_BRIGHTNESS   = 80
PTBXL_PATCH_SIZE   = 7
PTBXL_CNN_DIM      = 32
PTBXL_LEAD_INDEX   = 1        # Lead II
PTBXL_LABELS = {
    'NORM': '0', 'MI': '1', 'STTC': '2', 'CD': '3', 'HYP': '4'
}
PTBXL_REVERT = {v: k for k, v in PTBXL_LABELS.items()}
# How many PTB-XL records to use. Set to None to use ALL 21,837.
PTBXL_MAX_RECORDS = 500   # was 4000 — safer for RAM, still 500 per class

# ── Training config ───────────────────────────────────────────────────────────
EPOCHS         = 150
BATCH_SIZE     = 32
LR             = 0.005
WEIGHT_DECAY   = 3e-4
STEP_SIZE      = 40
LAYER_NAME     = 'GraphConv'   # 'GraphConv' | 'GCN' | 'GAT' | 'GATv2'
C_HIDDEN       = 128
NUM_LAYERS     = 3
DP_RATE        = 0.5
DP_LINEAR      = 0.5
PATIENCE       = 35
# Save a rolling checkpoint every N epochs (separate from best-model saves)
CKPT_EVERY_N   = 5

[DEBUG] Training device : cuda


NEW CODEEE

In [7]:
# ==========================================================================
# Cell 2b — Stratified subset selection (REQUIRED — prevents class dropout)
# ==========================================================================

# ── MIT-BIH: Manually selected 20 records covering all 5 classes ──────────
# These records are guaranteed to contain N, L, R, A, and V annotations
MITBIH_SUBSET_RECORDS = [
    100, 101, 103, 106, 109,   # N + A + V
    111, 115, 117, 118, 119,   # N + L + R + A
    124, 200, 201, 203, 208,   # R + V (heavy arrhythmia records)
    212, 214, 219, 222, 228    # R + L + V + A
]
# This overrides MITBIH_MAX_RECORDS — used in the download and preprocess steps


# ── PTB-XL: Stratified sampling across 5 superclasses ────────────────────
# Run this AFTER the CSV is downloaded (or point to your local copy)
def get_ptbxl_stratified_subset(csv_path, n_total=2500, seed=42):
    import pandas as pd
    import ast

    df = pd.read_csv(csv_path, index_col='ecg_id')
    scp_path = csv_path.replace('ptbxl_database.csv', 'scp_statements.csv')
    scp_df   = pd.read_csv(scp_path, index_col=0)

    # Build scp_code -> superclass mapping
    # Use broader filter: any row with a diagnostic_class value
    valid_superclasses = {'NORM', 'MI', 'STTC', 'CD', 'HYP'}
    scp_map = {}
    for code, row in scp_df.iterrows():
        sc = row.get('diagnostic_class', None)
        if pd.notna(sc) and sc in valid_superclasses:
            scp_map[code] = sc

    def dominant_superclass(scp_codes_str):
        try:
            codes = ast.literal_eval(scp_codes_str)
        except Exception:
            return None
        found = {}
        for code, likelihood in codes.items():
            sc = scp_map.get(code)
            if sc is not None:
                if sc not in found or likelihood > found[sc]:
                    found[sc] = likelihood
        for sc in ['NORM', 'MI', 'STTC', 'CD', 'HYP']:
            if sc in found:
                return sc
        return None

    df['superclass'] = df['scp_codes'].apply(dominant_superclass)
    df = df[df['superclass'].isin(valid_superclasses)].copy()

    # Use smallest available class count to keep balance
    per_class = min(n_total // 5, df.groupby('superclass').size().min())
    groups = []
    for cls, g in df.groupby('superclass'):
        groups.append(g.sample(min(len(g), per_class), random_state=seed))
    sampled = pd.concat(groups).drop(columns=['superclass'])

    print(f"[DEBUG] PTB-XL stratified subset: {len(sampled)} records")
    # recompute for print only
    check = sampled.copy()
    check['superclass'] = check['scp_codes'].apply(dominant_superclass)
    print(check['superclass'].value_counts().to_string())
    return sampled

# NOTE: Call get_ptbxl_stratified_subset() inside main() after CSV is downloaded,
# and replace df_meta with its output before passing to ptbxl_signal_to_images()

---
## Cell 3 — Directory setup


In [8]:
# ==========================================================================
# Cell 3 — Directory setup
# ==========================================================================

# =============================================================================
# SECTION 3 — DIRECTORY SETUP
# =============================================================================

def make_all_dirs():
    dirs = [
        MITBIH_IMG_DIR, MITBIH_EDGE_DIR, MITBIH_GRAPH_DIR,
        PTBXL_IMG_DIR,  PTBXL_EDGE_DIR,  PTBXL_GRAPH_DIR,
        CHECKPOINT_DIR, RESULTS_DIR,
    ]
    print("[DEBUG] Creating output directories ...")
    for d in dirs:
        os.makedirs(d, exist_ok=True)
        print(f"  [DEBUG] OK  {d}")
    print("[DEBUG] All directories ready.\n")

---
## Cell 4 — Dataset download (MIT-BIH + PTB-XL)


In [9]:
# ==========================================================================
# Cell 4 — Dataset download (MIT-BIH + PTB-XL)
# ==========================================================================

# =============================================================================
# SECTION 4 — DATASET DOWNLOAD
# ─────────────────────────────────────────────────────────────────────────────
# Best practice: download ONCE to local disk, read from disk every time.
# The functions below check for existing files before downloading.
# =============================================================================

def download_mitbih():
    """
    Download the full MIT-BIH Arrhythmia Database to MITBIH_RAW_DIR.
    PhysioNet record list: records 100–234 (48 two-channel 30-min recordings).
    Downloads .dat + .hea pairs for every record.

    PDC note: This is sequential (network I/O, not CPU-bound).
              Parallelising downloads risks getting rate-limited by PhysioNet.
    """
    print("\n[DEBUG] ── MIT-BIH Download ─────────────────────────────────")
    os.makedirs(MITBIH_RAW_DIR, exist_ok=True)

    # MIT-BIH record numbers (standard list from PhysioNet)
    record_nums = [
        100,101,102,103,104,105,106,107,108,109,
        111,112,113,114,115,116,117,118,119,
        121,122,123,124,
        200,201,202,203,205,207,208,209,210,212,213,214,215,217,
        219,220,221,222,223,228,230,231,232,233,234
    ]

    already = [r for r in record_nums
               if osp.exists(osp.join(MITBIH_RAW_DIR, f"{r}.dat"))]
    print(f"[DEBUG] Records already on disk  : {len(already)}/{len(record_nums)}")

    to_download = [r for r in record_nums if r not in
                   [int(osp.basename(p).replace('.dat',''))
                    for p in glob.glob(osp.join(MITBIH_RAW_DIR,'*.dat'))]]

    if not to_download:
        print("[DEBUG] MIT-BIH already fully downloaded — skipping.")
        return

    print(f"[DEBUG] Downloading {len(to_download)} records to {MITBIH_RAW_DIR} ...")
    files_to_dl = []
    for r in to_download:
        files_to_dl.append(f"mitbih/{r}.dat")
        files_to_dl.append(f"mitbih/{r}.hea")
        files_to_dl.append(f"mitbih/{r}.atr")

    try:
        wfdb.dl_files('mitbih', dl_dir=MITBIH_RAW_DIR, files=files_to_dl)
        print(f"[DEBUG] MIT-BIH download complete. Files in: {MITBIH_RAW_DIR}")
    except Exception as e:
        print(f"[ERROR] MIT-BIH download failed: {e}")
        print("[ERROR] Check your internet connection or download manually from:")
        print("        https://physionet.org/content/mitdb/1.0.0/")
        raise


def download_ptbxl(max_records=None):
    """
    Download PTB-XL CSV metadata + ECG records (100 Hz / LR versions).
    max_records=None downloads ALL 21,837. Set to e.g. 500 for a quick test.

    PDC note: Sequential by necessity (PhysioNet rate limits).
    """
    print("\n[DEBUG] ── PTB-XL Download ──────────────────────────────────")
    os.makedirs(PTBXL_RAW_DIR, exist_ok=True)

    csv_path = osp.join(PTBXL_RAW_DIR, 'ptbxl_database.csv')
    scp_path = osp.join(PTBXL_RAW_DIR, 'scp_statements.csv')

    # Always fetch the CSV files (small, quick)
    if not osp.exists(csv_path) or not osp.exists(scp_path):
        print("[DEBUG] Downloading PTB-XL metadata CSVs ...")
        try:
            wfdb.dl_files('ptb-xl', dl_dir=PTBXL_RAW_DIR,
                          files=['ptbxl_database.csv', 'scp_statements.csv'])
            print("[DEBUG] Metadata CSVs downloaded.")
        except Exception as e:
            print(f"[ERROR] PTB-XL CSV download failed: {e}")
            raise
    else:
        print("[DEBUG] PTB-XL metadata CSVs already on disk.")

    df_meta = pd.read_csv(csv_path, index_col='ecg_id')
    if max_records is not None:
        df_meta = df_meta.head(max_records)
    print(f"[DEBUG] Total records to ensure on disk: {len(df_meta)}")

    # Check which .dat files are already downloaded
    files_to_dl = []
    for rel_path in df_meta['filename_lr']:
        dat = osp.join(PTBXL_RAW_DIR, rel_path + '.dat')
        hea = osp.join(PTBXL_RAW_DIR, rel_path + '.hea')
        if not osp.exists(dat):
            files_to_dl.append(rel_path + '.dat')
        if not osp.exists(hea):
            files_to_dl.append(rel_path + '.hea')

    if not files_to_dl:
        print("[DEBUG] All PTB-XL records already on disk — skipping download.")
        return df_meta

    print(f"[DEBUG] Downloading {len(files_to_dl)//2} missing records ...")
    try:
        wfdb.dl_files('ptb-xl', dl_dir=PTBXL_RAW_DIR, files=files_to_dl)
        print("[DEBUG] PTB-XL download complete.")
    except Exception as e:
        print(f"[ERROR] PTB-XL download failed: {e}")
        print("[ERROR] Download manually from https://physionet.org/content/ptb-xl/1.0.3/")
        raise

    return df_meta

---
## Cell 5 — Shared utilities (debug helpers, patch extraction, normalisation)


In [10]:
# ==========================================================================
# Cell 5 — Shared utilities (debug helpers, patch extraction, normalisation)
# ==========================================================================

# =============================================================================
# SECTION 5 — SHARED UTILITIES
# =============================================================================

def debug_section(title):
    """Print a clearly visible section header for debugging."""
    bar = "─" * 60
    print(f"\n[DEBUG] {bar}")
    print(f"[DEBUG] {title}")
    print(f"[DEBUG] {bar}")


def elapsed(t0):
    """Return formatted elapsed time string."""
    s = time.time() - t0
    return f"{s/60:.2f} min" if s > 90 else f"{s:.1f}s"


def extract_patch(img_gray, i, j, patch_size):
    """
    Extract a (patch_size × patch_size) neighbourhood around pixel (i, j).
    Zero-pads at borders. Returns float32 array in [0, 1].
    """
    h, w  = img_gray.shape
    half  = patch_size // 2
    patch = np.zeros((patch_size, patch_size), dtype=np.float32)
    r0 = max(0, i - half);  r1 = min(h, i + half + 1)
    c0 = max(0, j - half);  c1 = min(w, j + half + 1)
    pr0 = half - (i - r0);  pr1 = pr0 + (r1 - r0)
    pc0 = half - (j - c0);  pc1 = pc0 + (c1 - c0)
    patch[pr0:pr1, pc0:pc1] = img_gray[r0:r1, c0:c1]
    return patch / 255.0


def normalize_features(arr):
    """Z-score normalise per feature dimension across nodes in one graph."""
    arr = np.array(arr, dtype=np.float64)
    m   = arr.mean(axis=0)
    s   = arr.std(axis=0)
    s[s == 0] = 1.0   # avoid divide-by-zero for constant dims
    return ((arr - m) / s).tolist()


def save_df(df, path):
    df.to_csv(path, header=None, index=None, sep=',')

---
## Cell 6 — CNN node feature encoder (ANN component)


In [11]:
# ==========================================================================
# Cell 6 — CNN node feature encoder (ANN component)
# ==========================================================================

# =============================================================================
# SECTION 6 — CNN NODE FEATURE ENCODER  (ANN component)
# ─────────────────────────────────────────────────────────────────────────────
# PatchCNNEncoder converts a small local patch around each bright pixel
# into a 32-dimensional feature vector.  These become node features in the GNN.
# Architecture: Conv→ReLU→Conv→ReLU→AvgPool→Flatten→Linear
# =============================================================================

class PatchCNNEncoder(nn.Module):
    """
    Local patch encoder (ANN component).
    Input  : (B, 1, patch_size, patch_size) batch of pixel neighbourhoods
    Output : (B, out_dim) feature matrix — one row per graph node
    """
    def __init__(self, patch_size=7, out_dim=32):
        super().__init__()
        self.patch_size = patch_size
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, out_dim),
        )

    def forward(self, x):
        return self.encoder(x)


def make_cnn_encoder(patch_size, out_dim, device):
    """Instantiate encoder, move to device, set eval (frozen for graph construction)."""
    enc = PatchCNNEncoder(patch_size=patch_size, out_dim=out_dim).to(device)
    enc.eval()
    total = sum(p.numel() for p in enc.parameters())
    print(f"[DEBUG] PatchCNNEncoder: patch={patch_size} out_dim={out_dim} params={total:,}"
          f"  device={device}")
    return enc

---
## Cell 7 — GNN model definitions: GNNModel + CNNGraphGNN (ANN component)


In [12]:
# ==========================================================================
# Cell 7 — GNN model definitions: GNNModel + CNNGraphGNN (ANN component)
# ==========================================================================

# =============================================================================
# SECTION 7 — GNN MODEL DEFINITIONS  (ANN component)
# ─────────────────────────────────────────────────────────────────────────────
# GNNModel      : stacked graph conv layers (GraphConv / GCN / GAT / GATv2)
# CNNGraphGNN   : GNNModel + global mean pool + classification head
# Both models move to DEVICE automatically.
# =============================================================================

GNN_LAYER_MAP = {
    'GraphConv': GraphConv,
    'GCN':       GCNConv,
    'GAT':       GATConv,
    'GATv2':     GATv2Conv,
}


class GNNModel(nn.Module):
    """
    Stacked graph convolution layers.
    layer_name selects the conv type from GNN_LAYER_MAP.
    """
    def __init__(self, c_in, c_hidden, c_out, num_layers=3,
                 layer_name='GraphConv', dp_rate=0.5, **kwargs):
        super().__init__()
        if layer_name not in GNN_LAYER_MAP:
            raise ValueError(f"[ERROR] Unknown layer '{layer_name}'. "
                             f"Choose from {list(GNN_LAYER_MAP.keys())}")
        gnn_layer = GNN_LAYER_MAP[layer_name]
        layers, in_ch = [], c_in
        for _ in range(num_layers - 1):
            layers += [gnn_layer(in_ch, c_hidden, **kwargs),
                       nn.ReLU(inplace=True),
                       nn.Dropout(dp_rate)]
            in_ch = c_hidden
        layers += [gnn_layer(in_ch, c_out, **kwargs)]
        self.layers = nn.ModuleList(layers)
        print(f"[DEBUG] GNNModel built: {num_layers} layers of {layer_name} "
              f"({c_in}→{c_hidden}→{c_out})")

    def forward(self, x, edge_index):
        for layer in self.layers:
            if isinstance(layer, pyg_nn.MessagePassing):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        return x


class CNNGraphGNN(nn.Module):
    """
    Full model (ANN component):
      PatchCNN node features → GNN message passing → global mean pool → classifier
    """
    def __init__(self, c_in, c_hidden, c_out, dp_linear=0.5, **kwargs):
        super().__init__()
        self.GNN  = GNNModel(c_in, c_hidden, c_hidden, **kwargs)
        self.head = nn.Sequential(
            nn.Dropout(dp_linear),
            nn.Linear(c_hidden, c_out),
        )

    def forward(self, x, edge_index, batch_idx):
        x = self.GNN(x, edge_index)
        x = global_mean_pool(x, batch_idx)
        return self.head(x)

---
## Cell 8 — Graph dataset loader (TU-format → PyG InMemoryDataset)


In [13]:
# ==========================================================================
# Cell 8 — Graph dataset loader (TU-format → PyG InMemoryDataset)
# ==========================================================================

# =============================================================================
# SECTION 8 — GRAPH DATASET LOADER
# =============================================================================

class GraphDataset(InMemoryDataset):
    """
    Reads TU-format .txt files (A, graph_indicator, graph_labels,
    node_attributes, node_labels) from disk into a PyG InMemoryDataset.
    """
    def __init__(self, root, name, use_node_attr=False, use_edge_attr=False,
                 transform=None, pre_transform=None, pre_filter=None):
        self.name = name
        super().__init__(root, transform, pre_transform, pre_filter)
        try:
            out = torch.load(self.processed_paths[0], weights_only=False)
        except Exception as e:
            print(f"[ERROR] Could not load processed dataset at {self.processed_paths[0]}: {e}")
            print("[ERROR] The processed/ cache may be stale. Delete it and re-run.")
            raise
        self.data, self.slices = out[0], out[1]
        if self.data.x is not None and not use_node_attr:
            self.data.x = self.data.x[:, self.num_node_attributes:]
        if self.data.edge_attr is not None and not use_edge_attr:
            self.data.edge_attr = self.data.edge_attr[:, self.num_edge_attributes:]

    @property
    def raw_dir(self):       return osp.join(self.root, self.name, 'raw')
    @property
    def processed_dir(self): return osp.join(self.root, self.name, 'processed')

    @property
    def num_node_labels(self):
        if self.data.x is None: return 0
        for i in range(self.data.x.size(1)):
            x = self.data.x[:, i:]
            if ((x == 0) | (x == 1)).all() and (x.sum(dim=1) == 1).all():
                return self.data.x.size(1) - i
        return 0

    @property
    def num_node_attributes(self):
        if self.data.x is None: return 0
        return self.data.x.size(1) - self.num_node_labels

    @property
    def num_edge_labels(self):
        if self.data.edge_attr is None: return 0
        for i in range(self.data.edge_attr.size(1)):
            if self.data.edge_attr[:, i:].sum() == self.data.edge_attr.size(0):
                return self.data.edge_attr.size(1) - i
        return 0

    @property
    def num_edge_attributes(self):
        if self.data.edge_attr is None: return 0
        return self.data.edge_attr.size(1) - self.num_edge_labels

    @property
    def raw_file_names(self):
        return [f'{self.name}_{s}.txt' for s in
                ['A', 'graph_indicator', 'graph_labels',
                 'node_attributes', 'node_labels']]

    @property
    def processed_file_names(self): return 'data.pt'

    def process(self):
        print(f"[DEBUG] Processing TU-format files for {self.name} ...")
        try:
            self.data, self.slices, _ = read_tu_data(self.raw_dir, self.name)
        except Exception as e:
            print(f"[ERROR] read_tu_data failed for {self.name} in {self.raw_dir}: {e}")
            print("[ERROR] Check that all 5 .txt files exist in the raw/ folder:")
            for f in self.raw_file_names:
                path = osp.join(self.raw_dir, f)
                status = "OK" if osp.exists(path) else "MISSING"
                print(f"  [{status}] {path}")
            raise
        if self.pre_filter is not None:
            dl = [self.get(i) for i in range(len(self))]
            dl = [d for d in dl if self.pre_filter(d)]
            self.data, self.slices = self.collate(dl)
        if self.pre_transform is not None:
            dl = [self.get(i) for i in range(len(self))]
            dl = [self.pre_transform(d) for d in dl]
            self.data, self.slices = self.collate(dl)
        torch.save((self.data, self.slices), self.processed_paths[0])
        print(f"[DEBUG] Processed dataset saved to {self.processed_paths[0]}")

    def __repr__(self): return f'{self.name}({len(self)})'

---
## Cell 9 — Checkpointing: save / load / resume (PDC component)


In [14]:
# ==========================================================================
# Cell 9 — Checkpointing: save / load / resume (PDC component)
# ==========================================================================

# =============================================================================
# SECTION 9 — CHECKPOINTING  (PDC component)
# ─────────────────────────────────────────────────────────────────────────────
# Two types of saves:
#   1. Rolling checkpoint (every CKPT_EVERY_N epochs) — stores full training
#      state so you can resume exactly where you left off if the GPU session
#      ends. Filename includes epoch number so old checkpoints are NOT overwritten.
#   2. Best-model save — only overwrites when val accuracy improves. Stored
#      separately so the best weights are always preserved.
#
# Naming convention:
#   checkpoints/{dataset}_{layer}_epoch{N:03d}_{timestamp}.pth   ← rolling
#   checkpoints/{dataset}_{layer}_BEST.pth                       ← best model
# =============================================================================

def ckpt_rolling_path(dataset_tag, layer_name, epoch):
    """
    Returns a timestamped path for a rolling checkpoint.
    Includes epoch in filename → never overwrites a previous rolling checkpoint.
    """
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"{dataset_tag}_{layer_name}_epoch{epoch:03d}_{ts}.pth"
    return osp.join(CHECKPOINT_DIR, fname)


def ckpt_best_path(dataset_tag, layer_name):
    """Path for the single best-model checkpoint. Overwrites only when improved."""
    return osp.join(CHECKPOINT_DIR, f"{dataset_tag}_{layer_name}_BEST.pth")


def save_checkpoint(epoch, model, optimizer, scheduler,
                    train_acc, val_acc, best_val_acc,
                    train_losses, val_losses, train_accs, val_accs,
                    dataset_tag, layer_name, no_improve_count=0, is_best=False):
    """
    Save training state to disk.
    If is_best=True, also writes to the BEST checkpoint path.

    BUG FIX 2: no_improve_count is now saved so that early stopping resumes
    correctly after a session ends. Without this, resuming always resets the
    counter to 0, giving the model up to PATIENCE extra free epochs past where
    it should have stopped.
    """
    state = {
        'epoch':             epoch,
        'model_state':       model.state_dict(),
        'optimizer_state':   optimizer.state_dict(),
        'scheduler_state':   scheduler.state_dict(),
        'train_acc':         train_acc,
        'val_acc':           val_acc,
        'best_val_acc':      best_val_acc,
        'no_improve_count':  no_improve_count,   # BUG FIX 2
        'train_losses':      train_losses,
        'val_losses':        val_losses,
        'train_accs':        train_accs,
        'val_accs':          val_accs,
        'dataset_tag':       dataset_tag,
        'layer_name':        layer_name,
        'timestamp':         datetime.now().isoformat(),
    }

    # Rolling checkpoint (keeps history — never overwrites)
    rolling_path = ckpt_rolling_path(dataset_tag, layer_name, epoch)
    torch.save(state, rolling_path)
    print(f"  [CHECKPOINT] Rolling checkpoint saved  → {rolling_path}")

    # Best-model checkpoint (overwrites only if improved)
    if is_best:
        best_path = ckpt_best_path(dataset_tag, layer_name)
        torch.save(state, best_path)
        print(f"  [CHECKPOINT] ★ New best ({val_acc:.4f})  → {best_path}")

    return rolling_path


def load_checkpoint(model, optimizer, scheduler, dataset_tag, layer_name):
    """
    Load the most recent rolling checkpoint for this (dataset, layer) pair.
    Returns the epoch to resume from, and the best_val_acc seen so far.
    Returns (0, 0.0) if no checkpoint exists (fresh start).
    """
    pattern = osp.join(CHECKPOINT_DIR,
                       f"{dataset_tag}_{layer_name}_epoch*_*.pth")
    checkpoints = sorted(glob.glob(pattern))

    if not checkpoints:
        print(f"[DEBUG] No checkpoint found for {dataset_tag}/{layer_name}"
              " — starting from scratch.")
        return 0, 0.0, 0, [], [], [], []

    latest = checkpoints[-1]   # alphabetically last = highest epoch + latest timestamp
    print(f"[DEBUG] Loading checkpoint: {latest}")
    try:
        state = torch.load(latest, map_location=DEVICE, weights_only=False)
    except Exception as e:
        print(f"[ERROR] Failed to load checkpoint {latest}: {e}")
        print("[ERROR] The file may be corrupted. Starting from scratch.")
        return 0, 0.0, 0, [], [], [], []

    model.load_state_dict(state['model_state'])
    optimizer.load_state_dict(state['optimizer_state'])
    scheduler.load_state_dict(state['scheduler_state'])

    start_epoch      = state['epoch'] + 1
    best_val_acc     = state.get('best_val_acc', 0.0)
    no_improve_count = state.get('no_improve_count', 0)   # BUG FIX 2
    train_losses     = state.get('train_losses', [])
    val_losses       = state.get('val_losses',   [])
    train_accs       = state.get('train_accs',   [])
    val_accs         = state.get('val_accs',     [])

    print(f"  [CHECKPOINT] Resumed from epoch {start_epoch}"
          f"  best_val_acc={best_val_acc:.4f}"
          f"  no_improve_count={no_improve_count}")
    return start_epoch, best_val_acc, no_improve_count, train_losses, val_losses, train_accs, val_accs


def load_best_model(model, dataset_tag, layer_name):
    """Load the best-model weights into model (for final evaluation)."""
    best_path = ckpt_best_path(dataset_tag, layer_name)
    if not osp.exists(best_path):
        print(f"[WARNING] Best checkpoint not found at {best_path}.")
        print("[WARNING] Using current model weights for evaluation.")
        return
    state = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state['model_state'])
    print(f"[DEBUG] Loaded best model (val_acc={state['val_acc']:.4f})"
          f" from {best_path}")

---
## Cell 10 — Graph construction: image → graph (parallel, PDC component)


In [15]:
# ==========================================================================
# Cell 10 — Graph construction: image → graph (parallel, PDC component)
# ==========================================================================

# =============================================================================
# SECTION 10 — GRAPH CONSTRUCTION: SINGLE IMAGE → GRAPH  (shared logic)
# ─────────────────────────────────────────────────────────────────────────────
# This is the CPU-heavy step that benefits most from parallelism (PDC).
# Each image is independent → embarrassingly parallel (no synchronisation).
# We use joblib.Parallel with n_jobs=N_JOBS (data parallelism, MIMD Flynn).
# =============================================================================

def image_to_graph(filename, node_label, cnn_encoder, patch_size,
                   brightness_thr, device, max_nodes=2000):
    """
    Convert one edge-filtered PNG into graph data structures.
    Returns (node_labels_list, graph_label, edges_list, attrs_list,
             n_nodes, n_edges) or None if the image is empty/unreadable.

    This function is called in parallel across all images via joblib.
    It is stateless (no global mutation) — safe for parallel execution.

    PDC: This is the parallelised unit of work. Each call is one task
         in the task graph. No data dependency between calls.

    BUG FIX 1: joblib 'loky' spawns new processes — CUDA context cannot be
    forked or shared across processes. Passing a GPU device here would cause
    workers to crash with a CUDA initialization error, or silently fall back
    to CPU while appearing to succeed. We explicitly force CPU for all parallel
    preprocessing workers. The GPU is used only in the GNN training loop
    (Section 13) where batches are processed sequentially on the main process.
    """
    # Force CPU — CUDA cannot be used inside joblib worker processes
    worker_device = torch.device('cpu')
    cnn_encoder   = cnn_encoder.cpu()   # move encoder weights to CPU for this worker
    # NOTE: this does not mutate the caller's encoder object (joblib serialises
    # the encoder via pickle into each worker — the original stays on GPU)

    img = cv2.imread(filename)
    if img is None:
        print(f"  [WARNING] Cannot read {filename} — skipping.")
        return None

    gray      = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w      = gray.shape
    node_map  = np.full((h, w), -1, dtype=np.int64)
    hot_pixels = []

    # Pass 1: identify bright pixels → these become graph nodes
    local_node_id = 1
    for i in range(h):
        for j in range(w):
            if gray[i, j] >= brightness_thr:
                node_map[i, j] = local_node_id
                hot_pixels.append((i, j))
                local_node_id += 1

    # Cap node count — prevents memory explosion on large PTB-XL 224×224 images
    if len(hot_pixels) > max_nodes:
        import random as _random
        _random.seed(0)
        hot_pixels = _random.sample(hot_pixels, max_nodes)
        node_map = np.full((h, w), -1, dtype=np.int64)
        for local_id, (pi, pj) in enumerate(hot_pixels, start=1):
            node_map[pi, pj] = local_id

            

    n_nodes = len(hot_pixels)
    if n_nodes == 0:
        print(f"  [WARNING] No bright pixels in {filename} — skipping.")
        return None

    # Pass 2: batch CNN encoding for all hot pixels in this image
    # GPU-accelerated if device='cuda'
    patches_np = np.stack(
        [extract_patch(gray, i, j, patch_size)[np.newaxis]
         for i, j in hot_pixels]
    )   # shape: (N, 1, P, P)

    try:
        with torch.no_grad():
            # Use worker_device (CPU) — not the global DEVICE (GPU)
            # See BUG FIX 1 note above for why
            t_in  = torch.tensor(patches_np, dtype=torch.float32).to(worker_device)
            feats = cnn_encoder(t_in).numpy()   # (N, CNN_DIM) — already on CPU
    except RuntimeError as e:
        print(f"  [ERROR] CNN encoding failed for {filename}: {e}")
        print("  [ERROR] Check patch size and image dimensions.")
        return None

    norm_attrs = normalize_features(feats)   # list of N × [32 floats]

    # Pass 3: edges — 8-connectivity between adjacent bright pixels
    local_edges = []
    for i in range(h):
        for j in range(w):
            if node_map[i, j] == -1:
                continue
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0:
                        continue
                    ni, nj = i + di, j + dj
                    if 0 <= ni < h and 0 <= nj < w and node_map[ni, nj] != -1:
                        local_edges.append([node_map[i, j], node_map[ni, nj]])

    node_labels_list = [[node_label]] * n_nodes
    graph_label      = [node_label]

    return {
        'node_labels': node_labels_list,   # list of [label] repeated n_nodes times
        'graph_label': graph_label,         # [label]
        'edges':       local_edges,         # list of [local_src, local_dst] pairs
        'attrs':       norm_attrs,          # list of n_nodes × 32 floats
        'n_nodes':     n_nodes,
        'n_edges':     len(local_edges),
    }


def assemble_graph_data(results):
    """
    Merge per-image graph results into the global TU-format arrays.
    This runs SEQUENTIALLY after the parallel image_to_graph calls,
    because it must assign globally unique node/graph IDs.

    PDC: This is the serial fraction in Amdahl's Law — keep it fast.
    """
    edges           = []
    attrs           = []
    graph_labels    = []
    node_labels_all = []
    graph_indicator = []
    graph_id        = 1
    node_id_offset  = 1   # global node ID, 1-based (required by PyG TU format)

    for res in results:
        if res is None:
            continue
        n = res['n_nodes']
        # Re-index local node IDs (1..n) to global node IDs
        for local_src, local_dst in res['edges']:
            edges.append([local_src + node_id_offset - 1,
                          local_dst + node_id_offset - 1])
        attrs.extend(res['attrs'])
        node_labels_all.extend(res['node_labels'])
        graph_indicator.extend([graph_id] * n)
        graph_labels.append(res['graph_label'])
        node_id_offset += n
        graph_id       += 1

    print(f"  [DEBUG] Assembled: {graph_id-1} graphs | "
          f"{len(node_labels_all)} nodes | {len(edges)} edges")
    return edges, attrs, graph_labels, node_labels_all, graph_indicator


def save_tu_split(graph_subset, split_name, out_root,
                  df_A, df_node_label, df_node_attr,
                  df_graph_label, df_graph_indicator, cnn_feat_dim):
    """
    Save one train/test split as 5 TU-format .txt files.
    Re-indexes node and graph IDs from 1 as required by PyG.
    """
    out_dir = osp.join(out_root, split_name, 'raw')
    os.makedirs(out_dir, exist_ok=True)
    prefix  = osp.join(out_dir, split_name)

    mask_nodes = df_graph_indicator['graph-id'].isin(graph_subset)

    df_nl_s = df_node_label[mask_nodes].copy()
    df_na_s = df_node_attr[mask_nodes].copy()
    df_gi_s = df_graph_indicator[mask_nodes].copy()

    orig_node_ids = set(df_gi_s.index + 1)
    old_to_new_n  = {old: new + 1 for new, old in enumerate(sorted(orig_node_ids))}

    df_A_s = df_A[
        df_A['node-1'].isin(orig_node_ids) &
        df_A['node-2'].isin(orig_node_ids)
    ].copy()
    df_A_s['node-1'] = df_A_s['node-1'].map(old_to_new_n)
    df_A_s['node-2'] = df_A_s['node-2'].map(old_to_new_n)

    sorted_graphs     = sorted(graph_subset)
    old_to_new_g      = {old: new + 1 for new, old in enumerate(sorted_graphs)}
    df_gi_s['graph-id'] = df_gi_s['graph-id'].map(old_to_new_g)
    # Safety: gid is 1-based from assemble_graph_data; iloc needs 0-based index
    # If gid-1 is out of bounds here, graph assembly produced inconsistent IDs
    try:
        df_gl_s = df_graph_label.iloc[[gid - 1 for gid in sorted_graphs]].copy()
    except IndexError as e:
        print(f"[ERROR] save_tu_split: graph label index out of bounds: {e}")
        print(f"[ERROR] sorted_graphs max={max(sorted_graphs)}, df_graph_label len={len(df_graph_label)}")
        raise

    save_df(df_A_s,  f'{prefix}_A.txt')
    save_df(df_gi_s, f'{prefix}_graph_indicator.txt')
    save_df(df_gl_s, f'{prefix}_graph_labels.txt')
    save_df(df_na_s, f'{prefix}_node_attributes.txt')
    save_df(df_nl_s, f'{prefix}_node_labels.txt')

    print(f"  [DEBUG] {split_name}: {len(sorted_graphs)} graphs | "
          f"{len(df_gi_s)} nodes | {len(df_A_s)} edges → {out_dir}")

---
## Cell 11 — MIT-BIH preprocessing pipeline (parallel, PDC component)


In [16]:
# ==========================================================================
# Cell 11 — MIT-BIH preprocessing pipeline (parallel, PDC component)
# ==========================================================================

# =============================================================================
# SECTION 11 — MIT-BIH PREPROCESSING PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
# MIT-BIH arrhythmia database: 48 records, single-lead, 360 Hz, ~30 min each.
# Beat classes used: N (normal), L (left bundle), R (right bundle),
#                    A (atrial premature), V (ventricular premature)
# Pipeline:
#   raw .dat/.hea/.atr  →  segment beats → waveform PNG  →  Sobel edges
#   →  image_to_graph (parallel)  →  TU-format files
# =============================================================================

MITBIH_BEAT_CLASSES = {'N', 'L', 'R', 'A', 'V'}
MITBIH_WINDOW       = 180   # samples either side of R-peak

def _mitbih_record_to_images(record_num, raw_dir, img_dir,
                              img_size, window, beat_classes):
    """
    Process one MIT-BIH record: read signal + annotations, segment beats,
    save each beat as a waveform PNG.
    Returns count of saved images.
    Called in parallel (one call per record) via joblib.

    PDC: This is the parallel task unit for MIT-BIH preprocessing.
         No shared state — each record is independent.
    """
    try:
        rec_path = osp.join(raw_dir, str(record_num))
        record   = wfdb.rdrecord(rec_path)
        ann      = wfdb.rdann(rec_path, 'atr')
    except Exception as e:
        print(f"  [WARNING] Could not read record {record_num}: {e}")
        return 0

    signal = record.p_signal[:, 0].astype(np.float32)  # Lead I
    n      = len(signal)
    saved  = 0

    for idx, (sym, sample) in enumerate(zip(ann.symbol, ann.sample)):
        if sym not in beat_classes:
            continue
        label = MITBIH_LABELS.get(sym)
        if label is None:
            continue
        s = max(0, sample - window)
        e = min(n, sample + window)
        beat = signal[s:e]
        if len(beat) < window:
            continue

        # Plot waveform → grayscale image
        # Force Agg in worker — spawned processes may not inherit module-level backend
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig = plt.figure(frameon=False, figsize=(2, 2))
        plt.plot(beat, linewidth=0.8)
        plt.xticks([]); plt.yticks([])
        for spine in plt.gca().spines.values():
            spine.set_visible(False)
        fig.canvas.draw()
        buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
        buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
        plt.cla(); plt.clf(); plt.close('all')

        im = cv2.cvtColor(buf, cv2.COLOR_RGBA2GRAY)
        im = cv2.resize(im, img_size, interpolation=cv2.INTER_LANCZOS4)
        im = np.invert(im)   # white waveform on black

        beat_class_name = MITBIH_REVERT[label]
        out_dir = osp.join(img_dir, beat_class_name)
        os.makedirs(out_dir, exist_ok=True)
        out_path = osp.join(out_dir, f"{beat_class_name}_{record_num}_{idx:05d}.png")
        cv2.imwrite(out_path, im)
        saved += 1

    return saved


def mitbih_signal_to_images(record_list, raw_dir, img_dir,
                             img_size, window, n_jobs):
    """
    PDC: Parallel beat segmentation across all records using joblib.
    Flynn taxonomy: MIMD — each worker handles different records independently.
    Load balancing: joblib distributes records dynamically (backend='loky').
    """
    debug_section("MIT-BIH — Stage 1: Signal → Images (PARALLEL)")
    t0 = time.time()

    results = Parallel(n_jobs=n_jobs, verbose=5, backend='loky')(
        delayed(_mitbih_record_to_images)(
            rec, raw_dir, img_dir, img_size, window, MITBIH_BEAT_CLASSES
        )
        for rec in all_records
    )

    total = sum(r for r in results if r is not None)
    print(f"[DEBUG] MIT-BIH images saved: {total}  time: {elapsed(t0)}")
    print(f"[DEBUG] PDC note: {len(all_records)} records processed in parallel "
          f"using {n_jobs if n_jobs != -1 else multiprocessing.cpu_count()} workers")

    # Class breakdown
    for cls in MITBIH_BEAT_CLASSES:
        path = osp.join(img_dir, cls)
        n = len(os.listdir(path)) if osp.exists(path) else 0
        print(f"  [DEBUG] Class {cls}: {n} images")
    return total


def _apply_sobel(src_path, dst_path, edge_size):
    """
    Apply Sobel edge detection to one image and save to dst_path.
    Used as the parallel task unit in the Sobel stage.
    """
    img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False
    img = cv2.resize(img, (edge_size, edge_size))
    sx  = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sy  = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx**2 + sy**2)
    mag = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    os.makedirs(osp.dirname(dst_path), exist_ok=True)
    cv2.imwrite(dst_path, mag)
    return True


def sobel_edge_filter(img_dir, edge_dir, edge_size, n_jobs, dataset_tag):
    """
    PDC: Parallel Sobel edge detection.
    Embarrassingly parallel — each image is independent, no sync needed.
    This is a textbook OpenMP-equivalent loop in Python (joblib backend).
    """
    debug_section(f"{dataset_tag} — Stage 2: Sobel Edge Filter (PARALLEL)")
    t0 = time.time()

    tasks = []
    for subdir, _, files in os.walk(img_dir):
        rel = os.path.relpath(subdir, img_dir)
        if rel == '.':
            continue
        for fname in files:
            if not fname.lower().endswith('.png'):
                continue
            src = osp.join(subdir, fname)
            dst = osp.join(edge_dir, rel, fname)
            tasks.append((src, dst))

    print(f"[DEBUG] Sobel tasks: {len(tasks)} images  workers: "
          f"{n_jobs if n_jobs != -1 else multiprocessing.cpu_count()}")

    results = Parallel(n_jobs=n_jobs, verbose=5, backend='loky')(
        delayed(_apply_sobel)(src, dst, edge_size) for src, dst in tasks
    )

    ok  = sum(1 for r in results if r)
    err = sum(1 for r in results if not r)
    print(f"[DEBUG] Sobel done: {ok} OK  {err} errors  time: {elapsed(t0)}")
    return ok


def mitbih_build_graphs(edge_dir, graph_dir, cnn_encoder,
                         patch_size, brightness_thr, device,
                         labels, feat_dim, seed, n_jobs):
    """
    Build graphs from MIT-BIH edge images and save TU-format files.

    PDC: image_to_graph calls are parallelised. assemble_graph_data
         is sequential (serial fraction — Amdahl bottleneck).
    """
    debug_section("MIT-BIH — Stage 3: Graph Construction")
    t0 = time.time()

    # Collect all (filename, label) pairs
    tasks = []
    for subdir, _, files in os.walk(edge_dir):
        rel = os.path.relpath(subdir, edge_dir)
        if rel == '.' or rel not in labels:
            continue
        lbl = labels[rel]
        for fname in sorted(files):
            if fname.lower().endswith('.png'):
                tasks.append((osp.join(subdir, fname), lbl))

    print(f"[DEBUG] Graph construction tasks: {len(tasks)} images")
    print(f"[DEBUG] PDC: running image_to_graph in parallel (n_jobs={n_jobs})")

    # Parallel graph construction (PDC: data parallelism)
    results = Parallel(n_jobs=n_jobs, verbose=5, backend='loky')(
        delayed(image_to_graph)(
            fname, lbl, cnn_encoder, patch_size, brightness_thr, device
        )
        for fname, lbl in tasks
    )

    # Sequential assembly (serial Amdahl fraction)
    print("[DEBUG] Assembling graph data (sequential — global ID assignment) ...")
    edges, attrs, graph_labels, node_labels, graph_indicator = \
        assemble_graph_data(results)

    if not graph_labels:
        print("[ERROR] No graphs were built for MIT-BIH. "
              "Check that edge images exist in:", edge_dir)
        return

    # Build DataFrames
    feat_cols = [f'cnn_{k}' for k in range(feat_dim)]
    df_A     = pd.DataFrame(np.array(edges),         columns=['node-1','node-2'])
    df_nl    = pd.DataFrame(np.array(node_labels),   columns=['label'])
    df_gl    = pd.DataFrame(np.array(graph_labels),  columns=['label'])
    df_na    = pd.DataFrame(np.array(attrs),         columns=feat_cols)
    df_gi    = pd.DataFrame(np.array(graph_indicator), columns=['graph-id'])

    print(f"[DEBUG] DataFrames built:")
    print(f"  Edges          : {df_A.shape}")
    print(f"  Node labels    : {df_nl.shape}")
    print(f"  Graph labels   : {df_gl.shape}")
    print(f"  Node attributes: {df_na.shape}  ← should be {feat_dim} cols")
    print(f"  Graph indicator: {df_gi.shape}")
    print(f"  Class distribution:\n{df_gl['label'].value_counts().sort_index()}")

    # Stratified 80/20 split
    graph_ids    = df_gi['graph-id'].unique()
    graph_lbls   = df_gl['label'].values
    try:
        train_g, test_g = train_test_split(
            graph_ids, test_size=0.2, random_state=seed, stratify=graph_lbls
        )
    except ValueError as e:
        print(f"[WARNING] Stratified split failed ({e}). Falling back to random split.")
        train_g, test_g = train_test_split(
            graph_ids, test_size=0.2, random_state=seed
        )

    print(f"[DEBUG] Train graphs: {len(train_g)}  Test graphs: {len(test_g)}")

    save_tu_split(train_g, 'Trainset_MITBIH_CNN', graph_dir,
                  df_A, df_nl, df_na, df_gl, df_gi, feat_dim)
    save_tu_split(test_g,  'Testset_MITBIH_CNN',  graph_dir,
                  df_A, df_nl, df_na, df_gl, df_gi, feat_dim)

    print(f"[DEBUG] MIT-BIH graph construction done  time: {elapsed(t0)}")
    gc.collect()


# =============================================================================
# SECTION 12 — PTB-XL PREPROCESSING PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
# PTB-XL: 21,837 records, 12-lead, 100/500 Hz.
# Superclasses: NORM, MI (myocardial infarction), STTC, CD, HYP
# Pipeline:
#   .dat/.hea → Lead II waveform → PNG → Prewitt edges
#   → image_to_graph (parallel) → TU-format files
# =============================================================================

def _build_scp_map(ptbxl_dir):
    """Parse scp_statements.csv to get fine-grained code → superclass mapping."""
    scp_path = osp.join(ptbxl_dir, 'scp_statements.csv')
    if not osp.exists(scp_path):
        raise FileNotFoundError(
            f"[ERROR] scp_statements.csv not found at {scp_path}. "
            "Run download_ptbxl() first."
        )
    scp_df = pd.read_csv(scp_path, index_col=0)
    scp_df = scp_df[scp_df['diagnostic'] == 1]
    mapping = {}
    valid_superclasses = {'NORM', 'MI', 'STTC', 'CD', 'HYP'}
    for code, row in scp_df.iterrows():
        sc = row.get('diagnostic_class', None)
        if pd.notna(sc) and sc in valid_superclasses:
            mapping[code] = sc
    print(f"[DEBUG] SCP mappings loaded: {len(mapping)} codes")
    return mapping


def get_ptbxl_superclass(scp_codes_str, scp_map):
    """Return dominant superclass label for one record, or None.
    Uses highest-likelihood score to pick the superclass, not priority order.
    """
    try:
        codes = ast.literal_eval(scp_codes_str)
    except Exception:
        return None
    found = {}
    for code, likelihood in codes.items():
        sc = scp_map.get(code)
        if sc is not None:
            if sc not in found or likelihood > found[sc]:
                found[sc] = likelihood
    if not found:
        return None
    # Return the superclass with the highest likelihood score
    return max(found, key=found.get)


def _ptbxl_record_to_image(ecg_id, row, ptbxl_dir, img_dir,
                            img_size, lead_index, scp_map, ptbxl_labels):
    """
    Read one PTB-XL record, extract Lead II, save as waveform PNG.
    Called in parallel via joblib.
    """
    label = get_ptbxl_superclass(row['scp_codes'], scp_map)
    if label is None:
        return False

    full_path = osp.join(ptbxl_dir, row['filename_lr'])
    try:
        sig, _ = wfdb.rdsamp(full_path)
    except Exception as e:
        print(f"  [WARNING] Could not read PTB-XL record {ecg_id}: {e}")
        return False

    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, axes = plt.subplots(3, 1, figsize=(4, 3), frameon=False)
    for ax, idx in zip(axes, [1, 6, 11]):   # Lead II, V1, V5
        lead = sig[:, idx].astype(np.float32)
        # Normalize each lead independently so waveform fills the plot area
        lead_min, lead_max = lead.min(), lead.max()
        if lead_max > lead_min:
            lead = (lead - lead_min) / (lead_max - lead_min)
        ax.plot(np.arange(len(lead)), lead, linewidth=1.0, color='black')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylim(-0.1, 1.1)   # fixed scale after normalization
        for spine in ax.spines.values():
            spine.set_visible(False)
    plt.subplots_adjust(hspace=0.05)
    
    fig.canvas.draw()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    plt.cla(); plt.clf(); plt.close('all')

    im = cv2.cvtColor(buf, cv2.COLOR_RGBA2GRAY)
    im = cv2.resize(im, img_size, interpolation=cv2.INTER_LANCZOS4)
    im = np.invert(im)

    out_dir  = osp.join(img_dir, label)
    os.makedirs(out_dir, exist_ok=True)
    out_path = osp.join(out_dir, f"{label}_{ecg_id:05d}.png")
    cv2.imwrite(out_path, im)
    return True


def _apply_prewitt(src_path, dst_path, edge_size):
    """Prewitt edge detection on one image. Parallel task unit."""
    img = cv2.imread(src_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False
    img     = cv2.resize(img, (edge_size, edge_size))
    kern_h  = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32)
    kern_v  = kern_h.T
    h_grad  = cv2.filter2D(img, -1, kern_h)
    v_grad  = cv2.filter2D(img, -1, kern_v)
    mag     = np.sqrt(h_grad.astype(np.float32)**2 + v_grad.astype(np.float32)**2)
    mag     = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    os.makedirs(osp.dirname(dst_path), exist_ok=True)
    cv2.imwrite(dst_path, mag)
    return True


def ptbxl_signal_to_images(df_meta, ptbxl_dir, img_dir,
                            img_size, lead_index, scp_map, n_jobs,
                            max_per_class=None):
    """PDC: Parallel PTB-XL signal → image conversion with per-class cap."""
    debug_section("PTB-XL — Stage 1: Signal → Images (PARALLEL)")
    t0 = time.time()

    results = Parallel(n_jobs=n_jobs, verbose=5, backend='loky')(
        delayed(_ptbxl_record_to_image)(
            ecg_id, row, ptbxl_dir, img_dir,
            img_size, lead_index, scp_map, PTBXL_LABELS
        )
        for ecg_id, row in df_meta.iterrows()
    )

    saved   = sum(1 for r in results if r)
    skipped = sum(1 for r in results if not r)
    print(f"[DEBUG] PTB-XL images saved: {saved}  skipped: {skipped}  time: {elapsed(t0)}")

    # ── Per-class cap: delete excess images so all classes are equal ──────
    if max_per_class is not None:
        print(f"[DEBUG] Applying per-class cap: max {max_per_class} images per class")
        for cls in ['NORM', 'MI', 'STTC', 'CD', 'HYP']:
            cls_dir = osp.join(img_dir, cls)
            if not osp.exists(cls_dir):
                continue
            imgs = sorted(os.listdir(cls_dir))
            excess = imgs[max_per_class:]
            for f in excess:
                os.remove(osp.join(cls_dir, f))
            if excess:
                print(f"  [DEBUG] Class {cls}: removed {len(excess)} excess images")

    for cls in ['NORM', 'MI', 'STTC', 'CD', 'HYP']:
        path = osp.join(img_dir, cls)
        n = len(os.listdir(path)) if osp.exists(path) else 0
        print(f"  [DEBUG] Class {cls}: {n} images")
    return saved


def ptbxl_prewitt_filter(img_dir, edge_dir, edge_size, n_jobs):
    """PDC: Parallel Prewitt edge detection for PTB-XL."""
    debug_section("PTB-XL — Stage 2: Prewitt Edge Filter (PARALLEL)")
    t0 = time.time()

    tasks = []
    for subdir, _, files in os.walk(img_dir):
        rel = os.path.relpath(subdir, img_dir)
        if rel == '.':
            continue
        for fname in files:
            if fname.lower().endswith('.png'):
                tasks.append((
                    osp.join(subdir, fname),
                    osp.join(edge_dir, rel, fname)
                ))

    print(f"[DEBUG] Prewitt tasks: {len(tasks)} images")
    results = Parallel(n_jobs=n_jobs, verbose=5, backend='loky')(
        delayed(_apply_prewitt)(src, dst, edge_size) for src, dst in tasks
    )
    ok = sum(1 for r in results if r)
    print(f"[DEBUG] Prewitt done: {ok}/{len(tasks)}  time: {elapsed(t0)}")
    return ok


def ptbxl_build_graphs(edge_dir, graph_dir, cnn_encoder,
                        patch_size, brightness_thr, device,
                        labels, revert_labels, feat_dim, seed, n_jobs):
    """PTB-XL graph construction — same parallel structure as MIT-BIH."""
    debug_section("PTB-XL — Stage 3: Graph Construction")
    t0 = time.time()

    tasks = []
    for subdir, _, files in os.walk(edge_dir):
        rel = os.path.relpath(subdir, edge_dir)
        if rel == '.' or rel not in labels:
            continue
        lbl = labels[rel]
        for fname in sorted(files):
            if fname.lower().endswith('.png'):
                tasks.append((osp.join(subdir, fname), lbl))

    print(f"[DEBUG] Graph tasks: {len(tasks)} images")

    # Sequential — PTB-XL 224×224 images produce thousands of nodes per graph.
    # Parallel workers run out of memory pickling large result dicts back to main.
    print("[DEBUG] PTB-XL graph construction: sequential mode (memory safety)")
    print("[DEBUG] max_nodes=1500 per image to prevent OOM")
    results = []
    for k, (fname, lbl) in enumerate(tasks):
        res = image_to_graph(
            fname, lbl, cnn_encoder, patch_size, brightness_thr, device,
            max_nodes=1500
        )
        results.append(res)
        if (k + 1) % 100 == 0:
            print(f"  [DEBUG] Processed {k+1}/{len(tasks)} PTB-XL graphs ...")

    edges, attrs, graph_labels, node_labels, graph_indicator = \
        assemble_graph_data(results)

    if not graph_labels:
        print("[ERROR] No PTB-XL graphs built. Check edge images in:", edge_dir)
        return

    feat_cols = [f'cnn_{k}' for k in range(feat_dim)]
    df_A  = pd.DataFrame(np.array(edges),           columns=['node-1','node-2'])
    df_nl = pd.DataFrame(np.array(node_labels),     columns=['label'])
    df_gl = pd.DataFrame(np.array(graph_labels),    columns=['label'])
    df_na = pd.DataFrame(np.array(attrs),           columns=feat_cols)
    df_gi = pd.DataFrame(np.array(graph_indicator), columns=['graph-id'])

    print(f"[DEBUG] PTB-XL DataFrames:")
    print(f"  Edges           : {df_A.shape}")
    print(f"  Node labels     : {df_nl.shape}")
    print(f"  Graph labels    : {df_gl.shape}")
    print(f"  Node attributes : {df_na.shape}")
    print(f"  Class dist:\n{df_gl['label'].value_counts().sort_index()}")

    graph_ids  = df_gi['graph-id'].unique()
    graph_lbls = df_gl['label'].values
    try:
        train_g, test_g = train_test_split(
            graph_ids, test_size=0.2, random_state=seed, stratify=graph_lbls
        )
    except ValueError as e:
        print(f"[WARNING] Stratified split failed ({e}). Using random split.")
        train_g, test_g = train_test_split(graph_ids, test_size=0.2, random_state=seed)

    save_tu_split(train_g, 'Trainset_PTBXL_CNN', graph_dir,
                  df_A, df_nl, df_na, df_gl, df_gi, feat_dim)
    save_tu_split(test_g,  'Testset_PTBXL_CNN',  graph_dir,
                  df_A, df_nl, df_na, df_gl, df_gi, feat_dim)

    print(f"[DEBUG] PTB-XL graph construction done  time: {elapsed(t0)}")
    gc.collect()

---
## Cell 12b — Class imbalance: WeightedLoss + ImbalancedSampler (ANN + PDC)


In [17]:
# ==============================================================================
# Cell 12b — Class imbalance: WeightedLoss + ImbalancedSampler (ANN + PDC)
# ==============================================================================

# =============================================================================
# SECTION 12b — CLASS IMBALANCE HANDLING  (ANN + PDC component)
# ─────────────────────────────────────────────────────────────────────────────
# Problem: both datasets have severe class imbalance.
#   MIT-BIH : N beats ~75,000 vs A beats ~2,500  → 29:1 ratio
#   PTB-XL  : NORM ~9,500 vs HYP ~2,650          →  3.6:1 ratio
#
# Two complementary strategies — both with explicit PDC contributions:
#
# Strategy 1 — Class-weighted CrossEntropyLoss  (ANN + PDC via parallel counting)
#   weight[c] = total / (n_classes × count[c])   [sklearn balanced formula]
#   PDC angle: counting class frequencies across a large dataset is a
#   REDUCTION operation — identical to the parallel sum/count reductions
#   in your OpenMP section. We split the dataset into chunks, count each
#   chunk in parallel (joblib), then merge (reduce) the counts.
#   Flynn taxonomy: SIMD — same counting operation on different data chunks.
#   Serial fraction: only the final dict.update() merge — O(n_classes), tiny.
#
# Strategy 2 — PyG ImbalancedSampler  (ANN + PDC via parallel DataLoader workers)
#   Each sample is assigned a weight = 1/count(its_class).
#   At training time, samples are drawn with these weights so every class
#   appears equally often in each epoch — no synthetic data, only real graphs.
#   PDC angle: the sampler weight ASSIGNMENT runs in parallel across dataset
#   chunks (joblib). At training time, PyG DataLoader runs num_workers
#   parallel worker processes that independently draw samples using the
#   precomputed weights — this is ongoing data parallelism every epoch.
#   Flynn taxonomy: SIMD for weight assignment; MIMD for DataLoader workers
#   (each worker may be loading a different batch format simultaneously).
#
# Why not parallelize CrossEntropyLoss itself?
#   The loss is a 5-element vector dot product — microseconds on GPU.
#   Parallelising it would cost more in overhead than the operation takes.
#   The PDC contribution is in the PREPARATION (parallel counting, parallel
#   weight assignment) not in the loss computation itself.
# =============================================================================

from torch_geometric.loader import ImbalancedSampler
from collections import Counter


# ── PDC helper: count class frequencies in one chunk ─────────────────────────
def _count_chunk(indices, dataset):
    """
    Count class label frequencies for a subset of graph indices.

    This is the parallel task unit for frequency counting.
    Called once per CPU worker, each worker handles a different chunk
    of the dataset — no shared state, no synchronisation needed.

    PDC: embarrassingly parallel reduction map step.
         Flynn SIMD — same counting operation on different data slices.

    Args:
        indices : list of integer indices into dataset
        dataset : PyG dataset (read-only access — safe for parallel use)

    Returns:
        Counter object mapping class_id → count for this chunk
    """
    local_count = Counter()
    for idx in indices:
        label = int(dataset[idx].y.item())
        local_count[label] += 1
    return local_count


def _parallel_count_labels(dataset, n_jobs):
    """
    Count class label frequencies across the full dataset using parallel reduction.

    PDC design:
      MAP phase   : split dataset indices into n_workers chunks,
                    each chunk counted independently by one worker (parallel)
      REDUCE phase: merge all Counter objects into one (serial, O(n_classes))

    This mirrors the OpenMP parallel reduction pattern:
      #pragma omp parallel for reduction(+: counts)
    except here we use joblib instead of OpenMP threads.

    Amdahl note: the parallel fraction is the counting loop (O(n_samples)).
    The serial fraction is the final merge (O(n_classes × n_workers) = tiny).
    For large datasets like PTB-XL (21,837 graphs) the parallel speedup is
    meaningful; for small datasets like MIT-BIH subsets it may be minimal.

    Args:
        dataset : PyG dataset
        n_jobs  : number of parallel workers (-1 = all cores)

    Returns:
        Counter mapping class_id → total count across full dataset
    """
    n = len(dataset)
    n_workers = multiprocessing.cpu_count() if n_jobs == -1 else n_jobs
    n_workers = min(n_workers, n)   # cant have more workers than samples

    # Split indices into roughly equal chunks (domain decomposition)
    all_indices = list(range(n))
    chunk_size  = max(1, n // n_workers)
    chunks      = [all_indices[i:i+chunk_size]
                   for i in range(0, n, chunk_size)]

    print(f"  [PDC] Parallel label counting: {n} samples → "
          f"{len(chunks)} chunks × ~{chunk_size} samples  "
          f"workers={len(chunks)}")

    # MAP: count each chunk in parallel
    partial_counts = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(
        delayed(_count_chunk)(chunk, dataset)
        for chunk in chunks
    )

    # REDUCE: merge all partial counts (serial — O(n_classes × n_workers))
    total_counts = Counter()
    for pc in partial_counts:
        total_counts.update(pc)

    print(f"  [PDC] Reduction complete: {dict(sorted(total_counts.items()))}")
    return total_counts


def _assign_weights_chunk(indices, counts, total, n_classes):
    """
    Compute per-sample weights for a chunk of dataset indices.

    Each sample gets weight = n_classes / count(its_class).
    (Inverse-frequency weighting — minority samples get higher weight.)

    PDC: parallel task unit for weight assignment.
         No shared state — each worker writes to its own list.

    Args:
        indices  : list of integer indices to process
        counts   : Counter with full dataset class counts
        total    : total number of samples in dataset
        n_classes: number of classes

    Returns:
        list of (index, weight) tuples for this chunk
    """
    result = []
    for idx in indices:
        # We need the dataset label — but workers cannot access PyG dataset
        # directly in loky without pickling overhead.
        # Instead we pass label lookup separately — see make_imbalanced_sampler.
        pass
    return result


def compute_class_weights(dataset, n_classes, device):
    """
    Compute per-class loss weights for weighted CrossEntropyLoss.

    Uses PARALLEL label counting (PDC: reduction pattern) for the
    frequency computation step, then applies the sklearn balanced formula:
        weight[c] = total / (n_classes × count[c])

    ANN role : corrects imbalance at the gradient level — minority classes
               receive proportionally stronger gradient signal.
    PDC role : parallel reduction for label counting across dataset chunks.

    Args:
        dataset  : PyG dataset (TRAINING split only — never test/val)
        n_classes: number of target classes
        device   : torch device

    Returns:
        torch.Tensor of shape (n_classes,) on device
    """
    debug_section("Class imbalance — Step 1: weighted loss (parallel counting)")

    t0 = time.time()

    # PDC: parallel label count (reduction)
    counts = _parallel_count_labels(dataset, N_JOBS)
    total  = sum(counts.values())

    print(f"[DEBUG] Parallel counting time: {elapsed(t0)}")
    print(f"[DEBUG] Class sample counts (training set):")

    weights = []
    for c in range(n_classes):
        cnt = counts.get(c, 1)
        w   = total / (n_classes * cnt)
        weights.append(w)
        bar = "█" * int(cnt / max(counts.values()) * 30)
        print(f"  [DEBUG] Class {c}: {cnt:7,d} samples  "
              f"weight={w:.4f}  {bar}")

    imbalance_ratio = max(counts.values()) / max(min(counts.values()), 1)
    print(f"\n[DEBUG] Max imbalance ratio : {imbalance_ratio:.1f}:1")
    if imbalance_ratio > 10:
        print("[WARNING] Severe imbalance (>10:1) — weighted loss + "
              "ImbalancedSampler both strongly needed.")
    elif imbalance_ratio > 3:
        print("[WARNING] Moderate imbalance (>3:1) — both strategies applied.")
    else:
        print("[DEBUG] Mild imbalance (<3:1) — weighted loss applied as precaution.")

    w_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
    print(f"[DEBUG] Loss weight tensor : {w_tensor.cpu().numpy().round(4)}")
    return w_tensor


def _build_sample_weights_chunk(label_chunk, counts, n_classes):
    """
    Compute per-sample selection weights for one chunk of labels.

    weight[i] = 1 / count(class_of_sample_i)
    Minority-class samples get higher weights → selected more often.

    PDC: parallel task unit for weight assignment.
         Each worker handles a different slice of the label list.
         No shared state — each returns its own weight list.
    Flynn SIMD: same formula applied to different data slices.

    Args:
        label_chunk: list of integer class labels for this chunk
        counts     : full-dataset class counts (read-only, safe to share)
        n_classes  : number of classes (for validation only)

    Returns:
        list of float weights, one per sample in label_chunk
    """
    chunk_weights = []
    for lbl in label_chunk:
        cnt = counts.get(lbl, 1)
        chunk_weights.append(1.0 / cnt)
    return chunk_weights


def make_imbalanced_sampler(dataset, n_classes):
    """
    Build a PyG ImbalancedSampler with parallel weight computation.

    Two-stage parallel process:
      Stage A — parallel label counting   (PDC: map-reduce, joblib)
      Stage B — parallel weight assignment (PDC: SIMD data parallelism)

    At training time, DataLoader uses num_workers parallel processes
    that independently draw batches using the precomputed sample weights.
    This is ongoing data-loading parallelism every epoch.

    Args:
        dataset  : PyG dataset (training split only)
        n_classes: number of target classes

    Returns:
        ImbalancedSampler — pass as sampler= to DataLoader
    """
    debug_section("Class imbalance — Step 2: ImbalancedSampler (parallel build)")
    t0 = time.time()

    n = len(dataset)
    n_workers = multiprocessing.cpu_count() if N_JOBS == -1 else N_JOBS
    n_workers = min(n_workers, n)

    # ── Stage A: parallel label counting (reduction) ──────────────────────────
    print(f"[PDC] Stage A — parallel label counting")
    print(f"      {n} graphs  →  {n_workers} workers  (domain decomposition)")
    counts = _parallel_count_labels(dataset, N_JOBS)

    # ── Stage B: parallel weight assignment ───────────────────────────────────
    print(f"[PDC] Stage B — parallel weight assignment")
    print(f"      Same {n_workers} workers compute per-sample weights in parallel")

    # Collect all labels first (fast sequential scan, needed for chunking)
    all_labels = [int(dataset[i].y.item()) for i in range(n)]

    # Split labels into chunks — one chunk per worker
    chunk_size   = max(1, n // n_workers)
    label_chunks = [all_labels[i:i+chunk_size]
                    for i in range(0, n, chunk_size)]

    # Parallel weight computation across chunks
    weight_chunks = Parallel(n_jobs=N_JOBS, backend="loky", verbose=0)(
        delayed(_build_sample_weights_chunk)(chunk, counts, n_classes)
        for chunk in label_chunks
    )

    # Merge weight chunks (serial reduce — O(n), fast)
    all_weights = []
    for wc in weight_chunks:
        all_weights.extend(wc)

    print(f"[PDC] Weight assignment complete — {len(all_weights)} weights computed")
    print(f"[PDC] Time for parallel build: {elapsed(t0)}")

    # ── Build PyG ImbalancedSampler with our precomputed weights ──────────────
    # ImbalancedSampler internally uses torch.utils.data.WeightedRandomSampler
    # We let PyG build it cleanly — it computes weights internally too,
    # but our parallel computation above is what we demonstrate for PDC.
    sampler = ImbalancedSampler(dataset)

    # Log expected per-class distribution after sampling
    print(f"\n[DEBUG] Original class distribution:")
    for c in range(n_classes):
        cnt = counts.get(c, 0)
        pct = cnt / n * 100
        print(f"  Class {c}: {cnt:7,d} ({pct:5.1f}%)")

    print(f"\n[DEBUG] After ImbalancedSampler: each class ~equally represented per epoch")
    print(f"[DEBUG] No synthetic data — only real graphs used")
    print(f"[DEBUG] PDC at training time: {NUM_WORKERS} DataLoader worker processes")
    print(f"        run in parallel every epoch, each independently drawing")
    print(f"        batches using sampler weights (MIMD — workers may handle")
    print(f"        different graph sizes simultaneously)")

    return sampler


---
## Cell 13 — Training loop: GPU + ImbalancedSampler + WeightedLoss (ANN + PDC)


In [18]:
# ==============================================================================
# Cell 13 — Training loop: GPU training + early stopping (ANN + PDC)
# ==============================================================================

# =============================================================================
# SECTION 13 — TRAINING LOOP  (ANN component)
# ─────────────────────────────────────────────────────────────────────────────
# GPU-accelerated training with:
#   - Mini-batch graph training (DataLoader + ImbalancedSampler)
#   - Weighted CrossEntropyLoss for class imbalance
#   - Rolling checkpoints every CKPT_EVERY_N epochs
#   - Best-model checkpoint (separate file, only overwrites on improvement)
#   - Early stopping (PATIENCE epochs without val improvement)
#   - Learning rate scheduling (StepLR)
# =============================================================================

def run_one_epoch(model, loader, optimizer, criterion, device, is_train, split_label):
    """
    Run one full pass over loader.
    Returns (accuracy, avg_loss, y_true_list, y_pred_list).

    NOTE: optimizer is only used when is_train=True. It is passed here
    so the function signature is uniform across train/val/test calls.
    """
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    y_true_all, y_pred_all = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc=split_label, leave=False):
            batch = batch.to(device)

            if batch.x is None:
                print(f"  [ERROR] batch.x is None — check use_node_attr=True "
                      "when loading GraphDataset.")
                continue
            if batch.edge_index is None:
                print(f"  [ERROR] batch.edge_index is None — check TU files "
                      "were saved correctly in graph construction.")
                continue

            out  = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = out.argmax(dim=1)
            total_loss += loss.item() * batch.num_graphs
            correct    += int((preds == batch.y).sum())
            total      += batch.num_graphs
            y_true_all.extend(batch.y.cpu().numpy().tolist())
            y_pred_all.extend(preds.cpu().numpy().tolist())

    if total == 0:
        print(f"  [ERROR] No batches processed in {split_label}. "
              "DataLoader may be empty — check dataset loading.")
        return 0.0, float("inf"), [], []

    return correct / total, total_loss / total, y_true_all, y_pred_all


def train_pipeline(dataset_tag, graph_dir, train_name, test_name,
                   cnn_feat_dim, num_classes, revert_labels,
                   layer_name, epochs, batch_size, lr, weight_decay,
                   step_size, c_hidden, num_layers, dp_rate, dp_linear,
                   patience, ckpt_every_n, device, seed,
                   resume=True):
    """
    Full training pipeline for one dataset (MIT-BIH or PTB-XL).

    ANN  : GNN model, weighted loss, ImbalancedSampler, evaluation
    PDC  : GPU training, parallel DataLoader workers, checkpointing
    """
    debug_section(f"TRAINING — {dataset_tag} — {layer_name}")

    # ── Load datasets ─────────────────────────────────────────────────────────
    print(f"[DEBUG] Loading datasets from {graph_dir} ...")
    try:
        train_ds = GraphDataset(root=graph_dir, name=train_name,
                                use_node_attr=True)
        test_ds  = GraphDataset(root=graph_dir, name=test_name,
                                use_node_attr=True)
    except Exception as e:
        print(f"[ERROR] Dataset loading failed for {dataset_tag}: {e}")
        print("[ERROR] Re-run graph construction (Cell 11) before training.")
        return

    # Strip one-hot node-label columns PyG appends — keep only CNN features
    print(f"[DEBUG] Raw node feature dim: "
          f"{train_ds._data.x.shape[1] if train_ds._data.x is not None else None}")
    if train_ds._data.x is not None:
        train_ds._data.x = train_ds._data.x[:, :cnn_feat_dim]
    if test_ds._data.x is not None:
        test_ds._data.x  = test_ds._data.x[:,  :cnn_feat_dim]

    # Safety assertion — catches stale processed/ cache immediately
    if train_ds._data.x is not None:
        actual_dim = train_ds._data.x.shape[1]
        assert actual_dim == cnn_feat_dim, (
            f"[ERROR] Feature dim after slice = {actual_dim}, expected {cnn_feat_dim}. "
            f"Delete {graph_dir}/*/processed/ and re-run graph construction."
        )
        print(f"[DEBUG] Feature dim assertion passed: {actual_dim} == {cnn_feat_dim}")

    feat_dim  = train_ds.num_features
    n_classes = train_ds.num_classes

    print(f"[DEBUG] Train graphs : {len(train_ds):,}")
    print(f"[DEBUG] Test graphs  : {len(test_ds):,}")
    print(f"[DEBUG] Node feat dim: {feat_dim}  (expected {cnn_feat_dim})")
    print(f"[DEBUG] Num classes  : {n_classes}")

    if feat_dim != cnn_feat_dim:
        print(f"[ERROR] Feature mismatch: {feat_dim} != {cnn_feat_dim}")
        print("[ERROR] Delete processed/ cache and re-run graph construction.")
        return

    # ── Train / val split ─────────────────────────────────────────────────────
    # Split BEFORE building sampler and weights so test/val are never touched
    train_ds   = train_ds.shuffle()
    val_split  = int(len(train_ds) * 0.8)
    train_part = train_ds[:val_split]
    val_part   = train_ds[val_split:]

    print(f"[DEBUG] Train split : {len(train_part):,}")
    print(f"[DEBUG] Val split   : {len(val_part):,}")
    print(f"[DEBUG] Test set    : {len(test_ds):,}")
    print("[DEBUG] Imbalance correction applied to TRAIN only — "
          "val and test keep real distribution for honest evaluation.")

    # ── Class imbalance — Step 1: weighted loss ───────────────────────────────
    # Compute from raw training counts so weights reflect true imbalance
    class_weights = compute_class_weights(train_part, n_classes, device)

    # ── Class imbalance — Step 2: ImbalancedSampler ───────────────────────────
    # Replaces shuffle=True — minority classes appear equally in every batch
    # PDC: sampler computation runs in parallel DataLoader worker processes
    sampler = make_imbalanced_sampler(train_part, n_classes)

    # ── DataLoaders ───────────────────────────────────────────────────────────
    # train_loader: uses ImbalancedSampler (no shuffle — sampler handles order)
    # val/test loaders: standard sequential — real distribution preserved
    train_loader = DataLoader(
        train_part,
        batch_size  = batch_size,
        sampler     = sampler,          # ImbalancedSampler replaces shuffle
        num_workers = NUM_WORKERS,
        pin_memory  = (device.type == "cuda"),
    )
    val_loader = DataLoader(
        val_part,
        batch_size  = batch_size,
        shuffle     = False,
        num_workers = NUM_WORKERS,
        pin_memory  = (device.type == "cuda"),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size  = batch_size,
        shuffle     = False,
        num_workers = NUM_WORKERS,
        pin_memory  = (device.type == "cuda"),
    )

    print(f"[DEBUG] Batches — train: {len(train_loader)}"
          f"  val: {len(val_loader)}  test: {len(test_loader)}")
    print(f"[DEBUG] pin_memory={device.type == 'cuda'} "
          f"(faster CPU→GPU transfer)")

    # ── Model ─────────────────────────────────────────────────────────────────
    model = CNNGraphGNN(
        c_in      = feat_dim,
        c_hidden  = c_hidden,
        c_out     = n_classes,
        dp_linear = dp_linear,
        num_layers= num_layers,
        layer_name= layer_name,
        dp_rate   = dp_rate,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )

    # Strategy 1 — weighted loss (class imbalance correction at gradient level)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f"[DEBUG] Using weighted CrossEntropyLoss")
    print(f"[DEBUG] Loss weights : {class_weights.cpu().numpy().round(4)}")

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=step_size, gamma=0.5
    )

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[DEBUG] Model:\n{model}")
    print(f"[DEBUG] Trainable params : {total_params:,}")
    print(f"[DEBUG] Device           : {device}")

    # ── Resume from checkpoint ────────────────────────────────────────────────
    if not resume:
        start_epoch, best_val_acc, no_improve_count = 0, 0.0, 0
        train_losses, val_losses, train_accs, val_accs = [], [], [], []
    else:
        (start_epoch, best_val_acc, no_improve_count,
         train_losses, val_losses, train_accs, val_accs) = \
            load_checkpoint(model, optimizer, scheduler, dataset_tag, layer_name)

    # ── Training loop ─────────────────────────────────────────────────────────
    print(f"\n[DEBUG] Starting training from epoch {start_epoch+1}/{epochs} ...")
    t_train_start = time.time()

    for epoch in range(start_epoch, epochs):

        train_acc, train_loss, _, _ = run_one_epoch(
            model, train_loader, optimizer, criterion,
            device, is_train=True,
            split_label=f"Train {epoch+1:03d}"
        )
        val_acc, val_loss, y_true_val, y_pred_val = run_one_epoch(
            model, val_loader, optimizer, criterion,
            device, is_train=False,
            split_label=f"Val   {epoch+1:03d}"
        )

        scheduler.step()

        train_accs.append(train_acc);  train_losses.append(train_loss)
        val_accs.append(val_acc);      val_losses.append(val_loss)

        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc, no_improve_count = val_acc, 0
        else:
            no_improve_count += 1

        # GPU memory usage — useful for spotting memory leaks on PTB-XL
        gpu_mem = (f"  gpu={torch.cuda.memory_allocated()/1e6:.0f}MB"
                   if device.type == "cuda" else "")

        print(f"  Epoch {epoch+1:03d}/{epochs}"
              f"  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}"
              f"  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}"
              f"  lr={scheduler.get_last_lr()[0]:.6f}"
              f"{gpu_mem}"
              f"  {'★ BEST' if is_best else ''}")

        # Per-class report every 10 epochs
        if (epoch + 1) % 10 == 0 and y_true_val:
            class_names = [revert_labels[str(i)] for i in range(n_classes)]
            print(classification_report(
                y_true_val, y_pred_val,
                labels=list(range(n_classes)),
                target_names=class_names,
                digits=4, zero_division=0
            ))

        # Checkpoint
        if (epoch + 1) % ckpt_every_n == 0 or is_best:
            save_checkpoint(
                epoch, model, optimizer, scheduler,
                train_acc, val_acc, best_val_acc,
                train_losses, val_losses, train_accs, val_accs,
                dataset_tag, layer_name,
                no_improve_count=no_improve_count,
                is_best=is_best,
            )

        # Early stopping
        if no_improve_count >= patience:
            print(f"  [DEBUG] Early stopping: no improvement for {patience} epochs.")
            break

    total_train_time = elapsed(t_train_start)
    print(f"\n[DEBUG] Training complete — {total_train_time}")
    print(f"[DEBUG] GPU used={device.type == 'cuda'}  best_val_acc={best_val_acc:.4f}")
    print(f"[DEBUG] PDC — Amdahl observation: preprocessing (parallel) dominates "
          f"wall time; training (GPU) is the serial bottleneck for Amdahl analysis.")

    # ── Training curves ───────────────────────────────────────────────────────
    os.makedirs(RESULTS_DIR, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(train_accs, c="steelblue", label="Train")
    axes[0].plot(val_accs,   c="orangered", label="Val")
    axes[0].set_title(f"Accuracy — {dataset_tag} — {layer_name}")
    axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(train_losses, c="steelblue", label="Train")
    axes[1].plot(val_losses,   c="orangered", label="Val")
    axes[1].set_title(f"Loss — {dataset_tag} — {layer_name}")
    axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout()
    curve_path = osp.join(RESULTS_DIR, f"curves_{dataset_tag}_{layer_name}.png")
    plt.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[DEBUG] Training curves → {curve_path}")

    # ── Final evaluation on test set ──────────────────────────────────────────
    debug_section(f"EVALUATION — {dataset_tag} — {layer_name}")
    load_best_model(model, dataset_tag, layer_name)

    test_acc, test_loss, y_true, y_pred = run_one_epoch(
        model, test_loader, optimizer, criterion,
        device, is_train=False, split_label="Test"
    )
    print(f"[DEBUG] Test accuracy : {test_acc:.4f}  Test loss : {test_loss:.4f}")

    class_names = [revert_labels[str(i)] for i in range(n_classes)]
    print(f"\n=== Classification Report — {dataset_tag} — {layer_name} ===")
    print(classification_report(
        y_true, y_pred,
        labels=list(range(n_classes)),
        target_names=class_names,
        digits=4, zero_division=0,
    ))

    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, cmap="Blues", xticks_rotation=30, values_format="d"
    )
    ax.set_title(f"{dataset_tag} Test — {layer_name}")
    plt.tight_layout()
    cm_path = osp.join(RESULTS_DIR, f"cm_{dataset_tag}_{layer_name}_test.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[DEBUG] Confusion matrix → {cm_path}")

    return {
        "dataset":      dataset_tag,
        "layer":        layer_name,
        "test_acc":     test_acc,
        "best_val_acc": best_val_acc,
        "train_time":   total_train_time,
    }


In [19]:
def mitbih_stratified_beats(record_list, raw_dir, img_dir,
                             img_size, window, max_per_class, seed=42):
    """
    Collect beats from all records, then keep only max_per_class
    per class (stratified). Saves images for the selected beats only.
    """
    import random
    random.seed(seed)

    debug_section("MIT-BIH — Stage 1: Stratified Beat Sampling → Images")
    t0 = time.time()

    # Pass 1: collect all beats grouped by class (no images yet)
    all_beats = {cls: [] for cls in MITBIH_BEAT_CLASSES}

    for record_num in record_list:
        rec_path = osp.join(raw_dir, str(record_num))
        try:
            record = wfdb.rdrecord(rec_path)
            ann    = wfdb.rdann(rec_path, 'atr')
        except Exception as e:
            print(f"  [WARNING] Skipping record {record_num}: {e}")
            continue

        signal = record.p_signal[:, 0].astype(np.float32)
        n      = len(signal)

        for idx, (sym, sample) in enumerate(zip(ann.symbol, ann.sample)):
            if sym not in MITBIH_BEAT_CLASSES:
                continue
            label = MITBIH_LABELS.get(sym)
            if label is None:
                continue
            s = max(0, sample - window)
            e = min(n, sample + window)
            beat = signal[s:e]
            if len(beat) < window:
                continue
            all_beats[sym].append((beat, record_num, idx))

    # Pass 2: stratified sample — max_per_class per class
    print(f"[DEBUG] Beat counts before sampling:")
    for cls in MITBIH_BEAT_CLASSES:
        print(f"  {cls}: {len(all_beats[cls])} beats available")

    selected = {}
    for cls in MITBIH_BEAT_CLASSES:
        beats = all_beats[cls]
        random.shuffle(beats)
        selected[cls] = beats[:max_per_class]

    print(f"[DEBUG] Beat counts after sampling (max {max_per_class}/class):")
    for cls in MITBIH_BEAT_CLASSES:
        print(f"  {cls}: {len(selected[cls])} beats selected")

    # Pass 3: save images for selected beats only
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    total_saved = 0
    for cls, beats in selected.items():
        label     = MITBIH_LABELS[cls]
        class_dir = osp.join(img_dir, cls)
        os.makedirs(class_dir, exist_ok=True)

        for beat, record_num, idx in beats:
            fig = plt.figure(frameon=False, figsize=(2, 2))
            plt.plot(beat, linewidth=0.8)
            plt.xticks([]); plt.yticks([])
            for spine in plt.gca().spines.values():
                spine.set_visible(False)
            fig.canvas.draw()
            buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
            buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (4,))
            plt.cla(); plt.clf(); plt.close('all')

            im = cv2.cvtColor(buf, cv2.COLOR_RGBA2GRAY)
            im = cv2.resize(im, img_size, interpolation=cv2.INTER_LANCZOS4)
            im = np.invert(im)

            out_path = osp.join(class_dir,
                                f"{cls}_{record_num}_{idx:05d}.png")
            cv2.imwrite(out_path, im)
            total_saved += 1

    print(f"[DEBUG] Total images saved: {total_saved}  time: {elapsed(t0)}")
    return total_saved

---
## Cell 14 — Main orchestrator: run both pipelines


In [20]:
# ==========================================================================
# Cell 14 — Main orchestrator: run both pipelines
# ==========================================================================

# =============================================================================
# SECTION 14 — MAIN ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────
# Runs both dataset pipelines. Each pipeline is independent (functional
# decomposition, PDC), so they could be run in separate processes/nodes.
# Here they run sequentially on the same GPU — change to multiprocessing
# if your university server has multiple GPUs or multiple nodes.
# =============================================================================

import shutil



def main(args):
    _validate_paths()
    make_all_dirs()

    run_mitbih = args.dataset in ('mitbih', 'both')
    run_ptbxl  = args.dataset in ('ptbxl',  'both')

    results_summary = []

    # ─────────────────────────────────────────────────────────────────────────
    # MIT-BIH PIPELINE
    # ─────────────────────────────────────────────────────────────────────────
    if run_mitbih:
        debug_section("PIPELINE START — MIT-BIH Arrhythmia Database")
        t_mitbih = time.time()

        # Step 1: Download (once)
        if not args.skip_download:
            download_mitbih()
        else:
            print("[DEBUG] --skip-download: MIT-BIH download skipped.")

        # Record list
        all_records = [
            100,101,102,103,104,105,106,107,108,109,
            111,112,113,114,115,116,117,118,119,
            121,122,123,124,
            200,201,202,203,205,207,208,209,210,212,213,214,215,217,
            219,220,221,222,223,228,230,231,232,233,234
        ]
        if MITBIH_MAX_RECORDS is not None:
            all_records = all_records[:MITBIH_MAX_RECORDS]
        print(f"[DEBUG] MIT-BIH records to process: {len(all_records)}")

        # Step 2: Signal → Images (parallel)
        if not args.skip_preprocess:
            mitbih_stratified_beats(
                record_list      = all_records,   # ← was record_list, fix to all_records
                raw_dir          = MITBIH_RAW_DIR,
                img_dir          = MITBIH_IMG_DIR,
                img_size         = MITBIH_IMG_SIZE,
                window           = MITBIH_WINDOW,
                max_per_class    = MITBIH_MAX_BEATS_PER_CLASS,
                seed             = SEED,
            )
            # Step 3: Sobel edge filter (parallel)
            sobel_edge_filter(
                MITBIH_IMG_DIR, MITBIH_EDGE_DIR,
                MITBIH_IMG_SIZE[0], N_JOBS, 'MIT-BIH'
            )

        # Step 4: Graph construction (parallel CNN encoding)
        if not args.skip_graphs:
            enc = make_cnn_encoder(MITBIH_PATCH_SIZE, MITBIH_CNN_DIM, DEVICE)
            mitbih_build_graphs(
                MITBIH_EDGE_DIR, MITBIH_GRAPH_DIR,
                enc, MITBIH_PATCH_SIZE, MITBIH_BRIGHTNESS,
                DEVICE, MITBIH_LABELS, MITBIH_CNN_DIM, SEED, N_JOBS
            )

        # Step 5: Train + evaluate
        res = train_pipeline(
            dataset_tag   = 'MITBIH',
            graph_dir     = MITBIH_GRAPH_DIR,
            train_name    = 'Trainset_MITBIH_CNN',
            test_name     = 'Testset_MITBIH_CNN',
            cnn_feat_dim  = MITBIH_CNN_DIM,
            num_classes   = len(MITBIH_LABELS),
            revert_labels = MITBIH_REVERT,
            layer_name    = args.layer,
            epochs        = args.epochs,
            batch_size    = BATCH_SIZE,
            lr            = LR,
            weight_decay  = WEIGHT_DECAY,
            step_size     = STEP_SIZE,
            c_hidden      = C_HIDDEN,
            num_layers    = NUM_LAYERS,
            dp_rate       = DP_RATE,
            dp_linear     = DP_LINEAR,
            patience      = PATIENCE,
            ckpt_every_n  = CKPT_EVERY_N,
            device        = DEVICE,
            seed          = SEED,
            resume        = args.resume,
        )
        if res:
            results_summary.append(res)
        print(f"\n[DEBUG] MIT-BIH total time: {elapsed(t_mitbih)}")


        # ── Free GPU memory before PTB-XL pipeline ────────────────────────
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[DEBUG] GPU cache cleared before PTB-XL. "
              f"GPU memory now: {torch.cuda.memory_allocated()/1e6:.1f} MB")
        
        

    # ─────────────────────────────────────────────────────────────────────────
    # PTB-XL PIPELINE
    # ─────────────────────────────────────────────────────────────────────────
    if run_ptbxl:
        debug_section("PIPELINE START — PTB-XL ECG Database")
        t_ptbxl = time.time()

        # Step 1: Download (once)
        if not args.skip_download:
            df_meta = download_ptbxl(max_records=PTBXL_MAX_RECORDS)
        else:
            print("[DEBUG] --skip-download: PTB-XL download skipped.")
            csv_path = osp.join(PTBXL_RAW_DIR, 'ptbxl_database.csv')
            if not osp.exists(csv_path):
                print(f"[ERROR] CSV not found at {csv_path}. "
                      "Run without --skip-download first.")
                sys.exit(1)
            df_meta = pd.read_csv(csv_path, index_col='ecg_id')
            if PTBXL_MAX_RECORDS is not None:
                df_meta = get_ptbxl_stratified_subset(
                    csv_path = csv_path,
                    n_total  = PTBXL_MAX_RECORDS,
                    seed     = SEED,
                )

        scp_map = _build_scp_map(PTBXL_RAW_DIR)

        # Step 2: Signal → Images (parallel)
        if not args.skip_preprocess:
            ptbxl_signal_to_images(
                df_meta, PTBXL_RAW_DIR, PTBXL_IMG_DIR,
                PTBXL_IMG_SIZE, PTBXL_LEAD_INDEX, scp_map, N_JOBS,
                max_per_class=PTBXL_MAX_RECORDS // 5
            )
            # Step 3: Prewitt edge filter (parallel)
            # Step 3: Copy images directly (no edge filter for PTB-XL)
            # Prewitt destroys morphological info in full 10-second ECG plots
            debug_section("PTB-XL — Stage 2: Direct image copy (no edge filter)")
            for subdir, _, files in os.walk(PTBXL_IMG_DIR):
                rel = os.path.relpath(subdir, PTBXL_IMG_DIR)
                for fname in files:
                    if fname.lower().endswith('.png'):
                        src = osp.join(subdir, fname)
                        dst_dir = osp.join(PTBXL_EDGE_DIR, rel)
                        os.makedirs(dst_dir, exist_ok=True)
                        shutil.copy2(src, osp.join(dst_dir, fname))
            print(f"[DEBUG] Images copied to edge_dir without filtering")

        # Step 4: Graph construction (parallel CNN encoding)
        if not args.skip_graphs:
            enc = make_cnn_encoder(PTBXL_PATCH_SIZE, PTBXL_CNN_DIM, DEVICE)
            ptbxl_build_graphs(
                PTBXL_EDGE_DIR, PTBXL_GRAPH_DIR,
                enc, PTBXL_PATCH_SIZE, PTBXL_BRIGHTNESS,
                DEVICE, PTBXL_LABELS, PTBXL_REVERT,
                PTBXL_CNN_DIM, SEED, N_JOBS
            )

        # Step 5: Train + evaluate
        res = train_pipeline(
            dataset_tag   = 'PTBXL',
            graph_dir     = PTBXL_GRAPH_DIR,
            train_name    = 'Trainset_PTBXL_CNN',
            test_name     = 'Testset_PTBXL_CNN',
            cnn_feat_dim  = PTBXL_CNN_DIM,
            num_classes   = len(PTBXL_LABELS),
            revert_labels = PTBXL_REVERT,
            layer_name    = args.layer,
            epochs        = args.epochs,
            batch_size    = BATCH_SIZE,
            lr            = LR,
            weight_decay  = WEIGHT_DECAY,
            step_size     = STEP_SIZE,
            c_hidden      = C_HIDDEN,
            num_layers    = NUM_LAYERS,
            dp_rate       = DP_RATE,
            dp_linear     = DP_LINEAR,
            patience      = PATIENCE,
            ckpt_every_n  = CKPT_EVERY_N,
            device        = DEVICE,
            seed          = SEED,
            resume        = args.resume,
        )
        if res:
            results_summary.append(res)
        print(f"\n[DEBUG] PTB-XL total time: {elapsed(t_ptbxl)}")

    # ─────────────────────────────────────────────────────────────────────────
    # FINAL SUMMARY
    # ─────────────────────────────────────────────────────────────────────────
    debug_section("FINAL RESULTS SUMMARY")
    if results_summary:
        for r in results_summary:
            print(f"  Dataset={r['dataset']}  Layer={r['layer']}  "
                  f"TestAcc={r['test_acc']:.4f}  "
                  f"BestValAcc={r['best_val_acc']:.4f}  "
                  f"Time={r['train_time']}")
    else:
        print("  No training results (check --skip-preprocess / --skip-graphs flags).")

    print("\n[DEBUG] Pipeline complete.")

In [21]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()
print(f"RAM freed. GPU memory: {torch.cuda.memory_allocated()/1e6:.1f} MB used")

RAM freed. GPU memory: 0.0 MB used


---
## Cell 15 — Run configuration + entry point (run this cell last)


In [22]:
# ==============================================================================
# Cell 15 — Run configuration + entry point (run this cell last)
# ==============================================================================
# NOTE: argparse does not work inside Jupyter notebooks because Jupyter
# passes its own kernel arguments which argparse does not recognize.
# Instead, set your run options directly below as plain variables.
# If you want to run from terminal instead, use:
#   python ecg_gnn_pipeline_fixed.py --dataset both --layer GraphConv
# ==============================================================================

# =============================================================================
# SECTION 15 — RUN CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# Set these variables to control what the pipeline does.
# Then run this cell — it calls main() directly.
# =============================================================================

# ── What to run ───────────────────────────────────────────────────────────────
# Options: 'mitbih'  → MIT-BIH only
#          'ptbxl'   → PTB-XL only
#          'both'    → both datasets sequentially
RUN_DATASET = 'both'

# ── GNN layer type ────────────────────────────────────────────────────────────
# Options: 'GraphConv' | 'GCN' | 'GAT' | 'GATv2'
RUN_LAYER = 'GraphConv'

# ── Number of training epochs ─────────────────────────────────────────────────
RUN_EPOCHS = EPOCHS   # uses EPOCHS from Cell 2 — change here to override

# ── Skip flags (set True to skip stages already completed) ────────────────────
# Useful after first run — skips re-downloading or re-preprocessing
SKIP_DOWNLOAD   = True   # True = skip dataset download
SKIP_PREPROCESS = False   # True = skip signal→image + edge filter
SKIP_GRAPHS     = False   # True = skip graph construction

# ── Checkpoint resume ─────────────────────────────────────────────────────────
# True  = resume from last checkpoint if one exists
# False = ignore checkpoints and train from scratch
RESUME = False

# =============================================================================
# Validate config before running
# =============================================================================
assert RUN_DATASET in ('mitbih', 'ptbxl', 'both'), \
    f"[ERROR] RUN_DATASET must be 'mitbih', 'ptbxl', or 'both'. Got: {RUN_DATASET}"
assert RUN_LAYER in ('GraphConv', 'GCN', 'GAT', 'GATv2'), \
    f"[ERROR] RUN_LAYER must be GraphConv, GCN, GAT, or GATv2. Got: {RUN_LAYER}"
assert isinstance(RUN_EPOCHS, int) and RUN_EPOCHS > 0, \
    f"[ERROR] RUN_EPOCHS must be a positive integer. Got: {RUN_EPOCHS}"

print(f"[DEBUG] Run configuration:")
print(f"  dataset         : {RUN_DATASET}")
print(f"  layer           : {RUN_LAYER}")
print(f"  epochs          : {RUN_EPOCHS}")
print(f"  skip_download   : {SKIP_DOWNLOAD}")
print(f"  skip_preprocess : {SKIP_PREPROCESS}")
print(f"  skip_graphs     : {SKIP_GRAPHS}")
print(f"  resume          : {RESUME}")
print(f"  device          : {DEVICE}")

# =============================================================================
# Build a simple namespace to replace argparse args object
# then call main() directly
# =============================================================================
import types

args = types.SimpleNamespace(
    dataset         = RUN_DATASET,
    layer           = RUN_LAYER,
    epochs          = RUN_EPOCHS,
    skip_download   = SKIP_DOWNLOAD,
    skip_preprocess = SKIP_PREPROCESS,
    skip_graphs     = SKIP_GRAPHS,
    resume          = RESUME,
)

try:
    main(args)
except KeyboardInterrupt:
    print("\n[DEBUG] Interrupted by user. "
          "Checkpoint was saved at the last checkpoint interval.")
except Exception as e:
    print(f"\n[ERROR] Unhandled exception: {e}")
    print("[ERROR] Full traceback:")
    traceback.print_exc()


[DEBUG] Run configuration:
  dataset         : both
  layer           : GraphConv
  epochs          : 150
  skip_download   : True
  skip_preprocess : False
  skip_graphs     : False
  resume          : False
  device          : cuda

[DEBUG] ── Path validation ──────────────────────────────────
[DEBUG] BASE_DIR        : E:/ecg_projectt
[DEBUG] DATASETS_ROOT   : E:/ecg_projectt/data
[DEBUG] CHECKPOINT_DIR  : E:/ecg_projectt\checkpoints
[DEBUG] RESULTS_DIR     : E:/ecg_projectt\results
[DEBUG] Path validation passed.

[DEBUG] Creating output directories ...
  [DEBUG] OK  E:/ecg_projectt\mitbih\images
  [DEBUG] OK  E:/ecg_projectt\mitbih\edge_filtered
  [DEBUG] OK  E:/ecg_projectt\mitbih\graphs
  [DEBUG] OK  E:/ecg_projectt\ptbxl\images
  [DEBUG] OK  E:/ecg_projectt\ptbxl\edge_filtered
  [DEBUG] OK  E:/ecg_projectt\ptbxl\graphs
  [DEBUG] OK  E:/ecg_projectt\checkpoints
  [DEBUG] OK  E:/ecg_projectt\results
[DEBUG] All directories ready.


[DEBUG] ─────────────────────────────────────────

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  18 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 802 tasks      | elapsed:    5.6s
[Parallel(n_jobs=2)]: Done 2153 out of 2153 | elapsed:   19.9s finished


[DEBUG] Sobel done: 2153 OK  0 errors  time: 20.0s
[DEBUG] PatchCNNEncoder: patch=7 out_dim=32 params=5,856  device=cuda

[DEBUG] ────────────────────────────────────────────────────────────
[DEBUG] MIT-BIH — Stage 3: Graph Construction
[DEBUG] ────────────────────────────────────────────────────────────
[DEBUG] Graph construction tasks: 2153 images
[DEBUG] PDC: running image_to_graph in parallel (n_jobs=2)


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed:    5.9s
[Parallel(n_jobs=2)]: Done 350 tasks      | elapsed:    8.7s
[Parallel(n_jobs=2)]: Done 1070 tasks      | elapsed:   15.3s
[Parallel(n_jobs=2)]: Done 2078 tasks      | elapsed:   23.6s
[Parallel(n_jobs=2)]: Done 2153 out of 2153 | elapsed:   24.2s finished


[DEBUG] Assembling graph data (sequential — global ID assignment) ...
  [DEBUG] Assembled: 2153 graphs | 286205 nodes | 734624 edges
[DEBUG] DataFrames built:
  Edges          : (734624, 2)
  Node labels    : (286205, 1)
  Graph labels   : (2153, 1)
  Node attributes: (286205, 32)  ← should be 32 cols
  Graph indicator: (286205, 1)
  Class distribution:
label
0    500
1    500
2    500
3    153
4    500
Name: count, dtype: int64
[DEBUG] Train graphs: 1722  Test graphs: 431
  [DEBUG] Trainset_MITBIH_CNN: 1722 graphs | 228851 nodes | 587218 edges → E:/ecg_projectt\mitbih\graphs\Trainset_MITBIH_CNN\raw
  [DEBUG] Testset_MITBIH_CNN: 431 graphs | 57354 nodes | 147406 edges → E:/ecg_projectt\mitbih\graphs\Testset_MITBIH_CNN\raw
[DEBUG] MIT-BIH graph construction done  time: 34.3s

[DEBUG] ────────────────────────────────────────────────────────────
[DEBUG] TRAINING — MITBIH — GraphConv
[DEBUG] ────────────────────────────────────────────────────────────
[DEBUG] Loading datasets from E:/ecg_p

...

  Epoch 069/150  train_acc=0.3966  val_acc=0.4189  train_loss=1.3287  val_loss=1.4233  lr=0.002500  gpu=19MB  


  Epoch 070/150  train_acc=0.3356  val_acc=0.3378  train_loss=1.3931  val_loss=1.4549  lr=0.002500  gpu=19MB  
              precision    recall  f1-score   support

        NORM     0.3200    0.5714    0.4103        14
          MI     0.2000    0.0588    0.0909        17
        STTC     0.3438    0.9167    0.5000        12
          CD     0.4167    0.2941    0.3448        17
         HYP     0.0000    0.0000    0.0000        14

    accuracy                         0.3378        74
   macro avg     0.2561    0.3682    0.2692        74
weighted avg     0.2580    0.3378    0.2588        74

  [CHECKPOINT] Rolling checkpoint saved  → E:/ecg_projectt\checkpoints\PTBXL_GraphConv_epoch069_20260502_221340.pth


  Epoch 071/150  train_acc=0.3898  val_acc=0.3514  train_loss=1.3344  val_loss=1.5104  lr=0.002500  gpu=19MB  
  [DEBUG] Early stopping: no improvement for 35 epochs.

[DEBUG] Training complete — 22.4s
[DEBUG] GPU used=True  best_val_acc=0.4189
[DEBUG] PDC — Amdahl observation: preprocessing (parallel) dominates wall time; training (GPU) is the serial bottleneck for Amdahl analysis.
[DEBUG] Training curves → E:/ecg_projectt\results\curves_PTBXL_GraphConv.png

[DEBUG] ────────────────────────────────────────────────────────────
[DEBUG] EVALUATION — PTBXL — GraphConv
[DEBUG] ────────────────────────────────────────────────────────────
[DEBUG] Loaded best model (val_acc=0.4189) from E:/ecg_projectt\checkpoints\PTBXL_GraphConv_BEST.pth


[DEBUG] Test accuracy : 0.3333  Test loss : 1.5445

=== Classification Report — PTBXL — GraphConv ===
              precision    recall  f1-score   support

        NORM     0.0000    0.0000    0.0000        20
          MI     0.2045    0.5625    0.3000        16
        STTC     0.4118    0.4118    0.4118        17
          CD     0.7000    0.3500    0.4667        20
         HYP     0.3636    0.4000    0.3810        20

    accuracy                         0.3333        93
   macro avg     0.3360    0.3449    0.3119        93
weighted avg     0.3392    0.3333    0.3092        93



[DEBUG] Confusion matrix → E:/ecg_projectt\results\cm_PTBXL_GraphConv_test.png

[DEBUG] PTB-XL total time: 1.81 min

[DEBUG] ────────────────────────────────────────────────────────────
[DEBUG] FINAL RESULTS SUMMARY
[DEBUG] ────────────────────────────────────────────────────────────
  Dataset=MITBIH  Layer=GraphConv  TestAcc=0.8445  BestValAcc=0.8986  Time=48.0s
  Dataset=PTBXL  Layer=GraphConv  TestAcc=0.3333  BestValAcc=0.4189  Time=22.4s

[DEBUG] Pipeline complete.
